# Signal-integrity convergence lab (private)

This notebook preserves the spatial-convergence, aligned-mesh, and archive-audit experiments for the differential-interconnect tutorial. It is **not** part of the published tutorial or the docs navigation, and it requires the IHP PDK, a local Palace runtime, `psutil`, `gmsh`, and the recorded run archives.

The exactly solvable TEM verification fixture now lives in `tests/palace/test_tem_line.py` and is run with `-m palace_local`.


In [ ]:
# Private lab flags. Leave both False unless you intend to launch fresh solves.
RUN_CONVERGENCE = False
RUN_RECOVERY = False

In [ ]:
import hashlib
import importlib.metadata as metadata
import json
import logging
import shutil
import subprocess
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skrf as rf
from IPython.display import display as show
from matplotlib.patches import Rectangle

from gsim.common.pdk import resolve_layer_stack
from gsim.palace import DrivenSim
from gsim.palace.signal_integrity import (
    check_network,
    complete_network,
    differential_delay,
    gap_arrival,
    interpolate_complex,
    line_network,
    mixed_mode,
    tdr,
)
from gsim.palace.stack import build_palace_layer_stack

EXECUTION = "recorded"  # recorded | preview | load | local | cloud
OUTPUT = (
    Path("..") if Path.cwd().name == "nbs" else Path()
) / "palace-signal-integrity-data"
PALACE_EXECUTABLE = str(
    Path.home() / ".cache/palacetoolkit/runtime/palace-cpu-v0.18.0/bin/palace"
)  # e.g. "/opt/palace/bin/palace"; None uses runtime discovery
PALACE_VERSION = "v0.18.0 (palacetoolkit CPU runtime; schema 1-6-0)"
LOCAL_PROCESSES = 4
PORTS = ("p1", "p2", "p3", "p4")
PAIRS = (("p1", "p2"), ("p3", "p4"))
Z_SE = 50.0

if EXECUTION not in {"recorded", "preview", "load", "local", "cloud"}:
    raise ValueError("Unknown execution mode")
if EXECUTION in {"local", "cloud"} and PALACE_VERSION == "UNRECORDED":
    raise ValueError("Record PALACE_VERSION before launching the experiment")

VERSIONS = {
    p: metadata.version(p)
    for p in ("gsim", "ihp-gdsfactory", "scikit-rf", "gdsfactory", "numpy", "scipy")
}
EXPECTED_VERSIONS = {"ihp-gdsfactory": "0.2.8", "scikit-rf": "2.1.0"}
VERSION_MISMATCHES = {
    package: (VERSIONS[package], expected)
    for package, expected in EXPECTED_VERSIONS.items()
    if VERSIONS[package] != expected
}
if VERSION_MISMATCHES and EXECUTION in {"local", "cloud"}:
    details = ", ".join(
        f"{package}={installed} (expected {expected})"
        for package, (installed, expected) in VERSION_MISMATCHES.items()
    )
    raise RuntimeError(
        "EM-generating runs require the recorded environment: " + details
    )
if VERSION_MISMATCHES:
    print(
        "Version note: recorded/preview analysis only; "
        + ", ".join(
            f"{package}={installed} (recorded with {expected})"
            for package, (installed, expected) in VERSION_MISMATCHES.items()
        )
    )
try:
    GSIM_REVISION = subprocess.check_output(  # noqa: S603 -- fixed read-only git arguments
        [shutil.which("git") or "/usr/bin/git", "rev-parse", "HEAD"],
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GSIM_REVISION = "unavailable"
try:
    IHP_INSTALL = metadata.distribution("ihp-gdsfactory").read_text("direct_url.json")
except (FileNotFoundError, OSError):
    IHP_INSTALL = None
show(pd.Series(VERSIONS, name="Installed version").to_frame())

In [ ]:
@dataclass(frozen=True)
class Geometry:
    length: float = 6000.0
    width: float = 14.0
    separation: float = 40.0  # signal center-to-center distance
    ground_width: float = 240.0
    gap_x: float = 2400.0  # slot center, measured from near reference plane
    gap_length: float = 120.0
    gap_y: float = 20.0
    gap_width: float = 36.0


GEO = Geometry()


def make_interconnect(g, signal_layer, return_layer, gap=False):
    if min(g.length, g.width, g.separation, g.ground_width) <= 0:
        raise ValueError("Geometry dimensions must be positive")
    if g.separation <= g.width or g.ground_width <= g.separation + g.width:
        raise ValueError("Signals must be disjoint and fit above the ground")
    c = gf.Component()
    for y, near, far in [
        (g.separation / 2, "p1", "p3"),
        (-g.separation / 2, "p2", "p4"),
    ]:
        c.add_polygon(
            [
                (0, y - g.width / 2),
                (g.length, y - g.width / 2),
                (g.length, y + g.width / 2),
                (0, y + g.width / 2),
            ],
            layer=signal_layer,
        )
        c.add_port(
            name=near,
            center=(0, y),
            width=g.width,
            orientation=180,
            layer=signal_layer,
            port_type="electrical",
        )
        c.add_port(
            name=far,
            center=(g.length, y),
            width=g.width,
            orientation=0,
            layer=signal_layer,
            port_type="electrical",
        )
    dbu = c.kcl.dbu

    def box(x0, y0, x1, y1):
        return gf.kdb.Box(*(round(v / dbu) for v in (x0, y0, x1, y1)))

    ground = gf.kdb.Region(box(0, -g.ground_width / 2, g.length, g.ground_width / 2))
    if gap:
        x0, x1 = g.gap_x - g.gap_length / 2, g.gap_x + g.gap_length / 2
        y0, y1 = g.gap_y - g.gap_width / 2, g.gap_y + g.gap_width / 2
        if not (
            0 < x0 < x1 < g.length
            and -g.ground_width / 2 < y0 < y1 < g.ground_width / 2
        ):
            raise ValueError("Slot must be strictly inside the ground plane")
        ground -= gf.kdb.Region(box(x0, y0, x1, y1))
    ground.merge()
    if ground.count() != 1:
        raise ValueError("Ground must remain one connected polygon")
    c.add_polygon(ground, layer=return_layer)
    return c


def show_layout(g):
    fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
    for ax, gap, name in zip(axes, [False, True], ["Intact", "Disrupted"], strict=True):
        ax.add_patch(
            Rectangle(
                (0, -g.ground_width / 2),
                g.length,
                g.ground_width,
                color="lightgray",
                label="Return plane (lower layer)",
            )
        )
        if gap:
            ax.add_patch(
                Rectangle(
                    (g.gap_x - g.gap_length / 2, g.gap_y - g.gap_width / 2),
                    g.gap_length,
                    g.gap_width,
                    color="white",
                    hatch="//",
                )
            )
        for y, near, far in [
            (g.separation / 2, "p1", "p3"),
            (-g.separation / 2, "p2", "p4"),
        ]:
            ax.add_patch(
                Rectangle(
                    (0, y - g.width / 2),
                    g.length,
                    g.width,
                    color="tab:blue",
                    alpha=0.65,
                )
            )
            ax.text(-60, y, near, ha="right", va="center")
            ax.text(g.length + 60, y, far, va="center")
        ax.set(
            xlim=(-250, g.length + 250), ylim=(-150, 150), ylabel="y (um)", title=name
        )
    axes[-1].set_xlabel("x (um); transverse scale enlarged for readability")
    fig.tight_layout()


show_layout(GEO)

In [ ]:
from ihp import PDK

PDK.activate()
IHP_REVISION = "d119d9ce0e9ad3092654a1cc9e1d0e18c1d6618f"
initial_stack = resolve_layer_stack(pdk=PDK)
SIGNAL_KEY, RETURN_KEY = "topmetal2", "metal1"
SIGNAL_LAYER = initial_stack.layers[SIGNAL_KEY].gds_layer
RETURN_LAYER = initial_stack.layers[RETURN_KEY].gds_layer
BULK = [
    {
        "name": "ild",
        "material": "sio2",
        "zmin": 0.0,
        "zmax": initial_stack.layers["passiv"].zmin,
    },
    {"name": "substrate", "material": "silicon", "zmin": -180.0, "zmax": 0.0},
]
MATERIAL_OVERRIDES = {
    "silicon": {
        "material_type": "semiconductor",
        "permittivity": 11.7,
        "conductivity": 10.0,
        "loss_tangent": 0.0,
    },
}
components = {
    name: make_interconnect(GEO, SIGNAL_LAYER, RETURN_LAYER, gap)
    for name, gap in [("intact", False), ("disrupted", True)]
}
resolved = resolve_layer_stack(components["intact"], pdk=PDK, dielectrics=BULK)
show(pd.DataFrame([asdict(resolved.layers[k]) for k in (SIGNAL_KEY, RETURN_KEY)]))
show(pd.DataFrame([asdict(d) for d in resolved.dielectrics]))
show(pd.DataFrame.from_dict(dict(resolved.materials), orient="index"))
show(pd.DataFrame.from_dict(MATERIAL_OVERRIDES, orient="index"))
print("Overlay:", resolved.overlay)
print("Reference planes: x=0 and x=", GEO.length, "um; no de-embedding")

In [ ]:
@dataclass(frozen=True)
class Settings:
    fmin: float = 0.25e9
    fmax: float = 200e9
    points: int = 800
    adaptive_tol: float = 1e-5
    refined_um: float = 6.0
    max_um: float = 100.0
    linear_solver: str = "MUMPS"
    linear_tolerance: float = 1e-8
    linear_max_iterations: int = 5
    real_preconditioner: bool = False
    order: int = 2
    air_um: float = 150.0


FULL_SETTINGS = Settings()
# Resource-bounded profile used for the recorded dense comparison.
BASE = replace(FULL_SETTINGS, order=1, refined_um=8.0, max_um=100.0)


def configure_sim(
    component, stack, signal_key, return_key, settings, outdir, material_overrides=None
):
    if signal_key == return_key or not {signal_key, return_key} <= set(stack.layers):
        raise ValueError(
            "Select distinct signal and return layers present in the stack"
        )
    if stack.layers[signal_key].z_bounds[0] <= stack.layers[return_key].z_bounds[1]:
        raise ValueError("This interlayer example requires signal above return")
    sim = DrivenSim()
    sim.set_output_dir(str(outdir))
    sim.set_geometry(component)
    sim.set_stack(build_palace_layer_stack(stack))
    for name, properties in (material_overrides or {}).items():
        sim.set_material(name, **properties)
    sim.set_airbox(
        margin_x=settings.air_um,
        margin_y=settings.air_um,
        z_above=settings.air_um,
        z_below=settings.air_um,
    )
    for name in PORTS:
        sim.add_port(
            name,
            from_layer=return_key,
            to_layer=signal_key,
            geometry="interlayer",
            impedance=Z_SE,
            excited=True,
        )
    sim.set_driven(
        fmin=settings.fmin,
        fmax=settings.fmax,
        num_points=settings.points,
        scale="linear",
        adaptive_tol=settings.adaptive_tol,
        adaptive_max_samples=80,
        reference_impedance=Z_SE,
    )
    sim.solver.order = settings.order
    sim.solver.linear.solver_type = settings.linear_solver
    sim.solver.linear.tolerance = settings.linear_tolerance
    sim.solver.linear.max_iterations = settings.linear_max_iterations
    sim.solver.linear.pc_mat_real = settings.real_preconditioner
    sim.solver.linear.complex_coarse_solve = not settings.real_preconditioner
    return sim


# Planning estimate only: actual delay is extracted from the solved baseline.
v_estimate = 1.5e8
frequency_step = (BASE.fmax - BASE.fmin) / (BASE.points - 1)
print(
    f"df={frequency_step / 1e9:g} GHz; periodic time span={1 / frequency_step * 1e12:g} ps"
)
print(
    f"Approximate sampling={1 / (2 * BASE.fmax) * 1e12:g} ps; "
    f"working resolution={2 / BASE.fmax * 1e12:g} ps"
)
print(f"Estimated gap round trip={2 * GEO.gap_x * 1e-6 / v_estimate * 1e12:g} ps")

In [ ]:
def load_recorded_case(label):
    candidates = []
    for path in OUTPUT.glob(label + "-*"):
        manifest_file = path / "manifest.json"
        if manifest_file.exists():
            manifest = json.loads(manifest_file.read_text())
            if manifest.get("status") == "complete":
                candidates.append((path, manifest))
    if not candidates:
        return None, None, None
    if len(candidates) != 1:
        raise ValueError(f"Ambiguous recorded case {label}; select an explicit archive")
    directory, manifest = candidates[0]
    data_path = directory / "network.npz"
    if hashlib.sha256(data_path.read_bytes()).hexdigest() != manifest["sha256"]:
        raise ValueError(f"Data hash mismatch for {label}")
    with np.load(data_path, allow_pickle=False) as saved:
        net = rf.Network(f=saved["f"], s=saved["s"], z0=saved["z0"], f_unit="Hz")
        net.port_names = saved["ports"].tolist()
    check_network(net, n_ports=4)
    return net, manifest, directory


def json_text(value):
    return json.dumps(value, sort_keys=True, indent=2, default=str)


def stack_record(stack):
    return {
        "pdk": stack.pdk_name,
        "overlay": stack.overlay,
        "notes": list(stack.notes),
        "layers": {k: asdict(v) for k, v in stack.layers.items()},
        "dielectrics": [asdict(d) for d in stack.dielectrics],
        "materials": dict(stack.materials),
    }


def validate_generated_config(config):
    ports = config["Boundaries"]["LumpedPort"]
    ids = [p.get("Excitation") for p in ports]
    if (
        len(ports) != 4
        or any(type(i) is not int for i in ids)
        or set(ids) != {1, 2, 3, 4}
    ):
        raise ValueError(f"Expected four independent excitations, got {ids}")
    if not all(np.isclose(p["R"], Z_SE) for p in ports):
        raise ValueError("Generated port impedances do not match the experiment")


def run_case(label, geometry, gap, settings, bulk=BULK):
    if EXECUTION == "recorded":
        recorded_label = {
            "intact": "refine8-intact",
            "disrupted": "refine8-disrupted",
        }.get(label)
        if recorded_label is None:
            return None  # The full proposed convergence suite was not completed.
        net, manifest, _ = load_recorded_case(recorded_label)
        if net is not None:
            if manifest["spec"]["geometry_um"] != asdict(geometry):
                raise ValueError("Recorded geometry differs from the tutorial controls")
            for key in (
                "fmin",
                "fmax",
                "points",
                "order",
                "refined_um",
                "max_um",
                "air_um",
            ):
                if manifest["spec"]["settings"][key] != getattr(settings, key):
                    raise ValueError(f"Recorded setting differs: {key}")
        return net
    component = make_interconnect(geometry, SIGNAL_LAYER, RETURN_LAYER, gap)
    stack = resolve_layer_stack(component, pdk=PDK, dielectrics=bulk)
    spec = {
        "schema": 1,
        "label": label,
        "geometry_um": asdict(geometry),
        "gap": gap,
        "settings": asdict(settings),
        "stack": stack_record(stack),
        "material_overrides": MATERIAL_OVERRIDES,
        "versions": VERSIONS,
        "gsim_revision": GSIM_REVISION,
        "ihp_source_revision": IHP_REVISION,
        "ihp_install": IHP_INSTALL,
        "palace_version": PALACE_VERSION,
        "ports": list(PORTS),
        "pairs": PAIRS,
        "z_se_ohm": Z_SE,
        "reference_planes_um": [0, geometry.length],
        "tdr": {"dc": "linear-real", "window": "hamming", "pad": 0},
    }
    signature = hashlib.sha256(json_text(spec).encode()).hexdigest()[:16]
    directory = OUTPUT / f"{label}-{signature}"
    manifest_path, data_path = directory / "manifest.json", directory / "network.npz"
    if EXECUTION == "preview":
        return None
    if manifest_path.exists() and data_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("status") == "complete":
            if json_text(manifest["spec"]) != json_text(spec):
                raise ValueError("Manifest mismatch")
            if hashlib.sha256(data_path.read_bytes()).hexdigest() != manifest["sha256"]:
                raise ValueError("Saved network hash mismatch")
            with np.load(data_path, allow_pickle=False) as saved:
                net = rf.Network(
                    f=saved["f"], s=saved["s"], z0=saved["z0"], f_unit="Hz"
                )
                net.port_names = saved["ports"].tolist()
            check_network(net, n_ports=4)
            return net
    if EXECUTION == "load":
        raise FileNotFoundError(f"No completed matching run in {directory}")
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {"status": "incomplete", "spec": spec}
    manifest_path.write_text(json_text(manifest))
    try:
        sim = configure_sim(
            component,
            stack,
            SIGNAL_KEY,
            RETURN_KEY,
            settings,
            directory,
            MATERIAL_OVERRIDES,
        )
        sim.mesh(
            preset="default",
            refined_mesh_size=settings.refined_um,
            max_mesh_size=settings.max_um,
            planar_conductors=False,
            fmax=settings.fmax,
            threads=1,
            surface_threads=1,
        )
        config_path = sim.write_config()
        config = json.loads(Path(config_path).read_text())
        validate_generated_config(config)
        run_logger = logging.getLogger("gsim.palace.base")
        log_handler = logging.FileHandler(directory / "solver.log")
        previous_level = run_logger.level
        run_logger.setLevel(logging.INFO)
        run_logger.addHandler(log_handler)
        try:
            if EXECUTION == "local":
                result = sim.run_local(
                    palace_executable=PALACE_EXECUTABLE,
                    num_processes=LOCAL_PROCESSES,
                    num_threads=2,
                )
            else:
                result = sim.run(check_cache=True)
        finally:
            run_logger.removeHandler(log_handler)
            log_handler.close()
            run_logger.setLevel(previous_level)
        from gsim.palace.validation import validate_solution

        evidence = validate_solution(
            (directory / "solver.log").read_text(),
            estimator_disabled=config["Solver"]["Linear"].get("EstimatorMaxIts") == 0,
            expected_excitations=[1, 2, 3, 4] if settings.adaptive_tol > 0 else [],
            exit_code=0 if EXECUTION == "local" else None,
        )
        manifest["solver_evidence"] = asdict(evidence)
        net = complete_network(result, ports=PORTS, z_se=Z_SE)
        target_f = np.linspace(settings.fmin, settings.fmax, settings.points)
        np.testing.assert_allclose(net.f, target_f, rtol=1e-8, atol=1)
        np.savez_compressed(
            data_path, f=net.f, s=net.s, z0=net.z0, ports=np.array(net.port_names)
        )
        net.write_touchstone(filename=str(directory / "network"))
        manifest.update(
            status="complete",
            sha256=hashlib.sha256(data_path.read_bytes()).hexdigest(),
            config=str(config_path),
        )
        manifest_path.write_text(json_text(manifest))
    except Exception as exc:
        manifest.update(status="failed", error=f"{type(exc).__name__}: {exc}")
        manifest_path.write_text(json_text(manifest))
        raise
    else:
        return net


networks = {
    label: run_case(label, GEO, gap, BASE)
    for label, gap in [("intact", False), ("disrupted", True)]
}
EM_READY = all(net is not None for net in networks.values())
print(
    "Base EM runs complete."
    if EM_READY
    else "EM results INCOMPLETE: preview only; analytic checks are not EM validation."
)

In [ ]:
def db(value):
    return 20 * np.log10(np.maximum(np.abs(value), 1e-7))


if EM_READY:
    mixed = {label: mixed_mode(net)[1] for label, net in networks.items()}
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
    for ax, (title, i, j) in zip(
        axes.flat,
        [
            ("Reflection S11", 0, 0),
            ("Transmission S31", 2, 0),
            ("NEXT S21", 1, 0),
            ("FEXT S41", 3, 0),
        ],
        strict=True,
    ):
        for label, net in networks.items():
            ax.plot(net.f / 1e9, db(net.s[:, i, j]), label=label)
        ax.set(title=title, xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
        ax.grid()
        ax.legend()
    fig.tight_layout()

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    for ax, (block, row, col) in zip(
        axes.flat, [("dd", 0, 0), ("dc", 0, 2), ("cd", 2, 0), ("cc", 2, 2)], strict=True
    ):
        for label, net in mixed.items():
            style = "-" if label == "intact" else "--"
            for k, (i, j) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
                ax.plot(
                    net.f / 1e9,
                    db(net.s[:, row + i, col + j]),
                    ls=style,
                    color=colors[k],
                    label=f"{label}: S{block}{i + 1}{j + 1}",
                )
        ax.set(title=f"S{block}", xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
        ax.grid()
        ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    show(
        pd.DataFrame(
            [
                {
                    "case": label,
                    "max |S-S^T|": np.max(np.abs(net.s - net.s.transpose(0, 2, 1))),
                    "max singular value": np.linalg.svd(net.s, compute_uv=False).max(),
                    "max mode conversion (linear)": max(
                        np.abs(mixed[label].s[:, :2, 2:]).max(),
                        np.abs(mixed[label].s[:, 2:, :2]).max(),
                    ),
                }
                for label, net in networks.items()
            ]
        )
    )
else:
    print("Frequency-domain EM plots pending completed runs.")

## 10. Convergence: distinguish a resolved effect from a numerical artifact

Enable `RUN_CONVERGENCE` to run the matrix below. Each row runs both geometries.
The three meshes test spatial refinement; higher field order tests the launch;
the larger airbox tests the absorbing boundary; the deeper silicon region tests
substrate truncation; the tighter adaptive sweep tests frequency interpolation.
Direct solves at nine frequencies independently check that interpolation.

The finest common setup is the comparison reference, not ground truth. We use the
largest observed complex-S change as an **empirical numerical uncertainty indicator**
$\epsilon_S$. It is not a continuum error bound. A corresponding indicator for TDR
is the largest reflected-step change $\epsilon_\rho$ across these studies.

Acceptance criteria chosen **before** examining EM results:

* maximum study difference $\epsilon_S<0.01$ in absolute complex S;
* baseline mode conversion no greater than $3\epsilon_S$;
* gap-induced mixed-mode conversion change greater than $3\epsilon_S$;
* gap reflection detectable above $3\epsilon_\rho$;
* slot arrival and moved-slot delay shift agree with the propagation estimate
  within the working time resolution, with events separated from the launches.

The factors of three are pragmatic margins, not statistical confidence intervals.
An inconclusive/failed check calls for refinement or a better-resolved geometry,
not relaxed thresholds. The two gap locations must also be distinguishable in time.

In [ ]:
FINE = replace(BASE, refined_um=2.5, max_um=50)
STUDIES = {
    "coarse": replace(BASE, refined_um=10, max_um=150),
    "medium": BASE,
    "fine": FINE,
    "order3": replace(FINE, order=3),
    "airbox": replace(FINE, air_um=300),
    "substrate": FINE,
    "sweep": replace(FINE, adaptive_tol=1e-6),
    "direct": replace(FINE, points=9, adaptive_tol=0),
}
GAP_SHIFT = 1200.0
study_networks = {}
if RUN_CONVERGENCE:
    for study, settings in STUDIES.items():
        if study == "medium" and EM_READY:
            study_networks[study] = networks
            continue
        bulk = (
            [dict(d, zmin=-360.0) if d["name"] == "substrate" else d for d in BULK]
            if study == "substrate"
            else BULK
        )
        study_networks[study] = {
            label: run_case(f"{study}-{label}", GEO, gap, settings, bulk)
            for label, gap in [("intact", False), ("disrupted", True)]
        }
    moved = run_case(
        "moved-slot", replace(GEO, gap_x=GEO.gap_x + GAP_SHIFT), True, FINE
    )
else:
    moved = None


def interpolate_complex(net, frequencies):
    # Used only to compare direct anchors with the reported dense sweep;
    # this never fabricates a TDR spectrum from sparse anchors.
    return np.stack(
        [
            np.interp(frequencies, net.f, net.s[:, i, j].real)
            + 1j * np.interp(frequencies, net.f, net.s[:, i, j].imag)
            for i in range(4)
            for j in range(4)
        ],
        axis=-1,
    ).reshape(-1, 4, 4)


STUDY_READY = (
    bool(study_networks)
    and moved is not None
    and all(n is not None for pair in study_networks.values() for n in pair.values())
)
if STUDY_READY:
    reference = {label: mixed_mode(n)[1] for label, n in study_networks["fine"].items()}
    rows = []
    for study, pair in study_networks.items():
        for label, n in pair.items():
            mm = mixed_mode(n)[1]
            ref = reference[label]
            err_s = np.max(np.abs(mm.s - interpolate_complex(ref, mm.f)))
            err_rho = np.nan
            if study != "direct":
                t0, r0, _ = tdr(ref.f, ref.s[:, 0, 0], 2 * Z_SE)
                t1, r1, _ = tdr(mm.f, mm.s[:, 0, 0], 2 * Z_SE)
                np.testing.assert_allclose(t0, t1)
                err_rho = np.max(np.abs(r1 - r0))
            rows.append(
                {
                    "study": study,
                    "geometry": label,
                    "max complex S change": err_s,
                    "max reflected-step change": err_rho,
                }
            )
    convergence = pd.DataFrame(rows)
    show(convergence)
    eps_s = max(convergence["max complex S change"].max(), 1e-12)
    eps_rho = max(convergence["max reflected-step change"].max(), 1e-12)
    intact, disrupted = reference["intact"], reference["disrupted"]

    def conversion(s):
        return np.concatenate([s[:, :2, 2:].reshape(-1), s[:, 2:, :2].reshape(-1)])

    baseline_conversion = np.max(np.abs(conversion(intact.s)))
    conversion_change = np.max(np.abs(conversion(disrupted.s - intact.s)))
    tau = differential_delay(intact)
    predicted = 2 * tau * GEO.gap_x / GEO.length
    moved_predicted = 2 * tau * (GEO.gap_x + GAP_SHIFT) / GEO.length
    resolution_s = 2 / FINE.fmax
    arrival = gap_arrival(intact, disrupted, tau, resolution_s, 3 * eps_rho)
    moved_mm = mixed_mode(moved)[1]
    moved_arrival = gap_arrival(intact, moved_mm, tau, resolution_s, 3 * eps_rho)
    expected_shift = moved_predicted - predicted
    separation_ok = (
        min(predicted, 2 * tau - moved_predicted) > 3 * resolution_s
        and expected_shift > resolution_s
    )
    checks = {
        "absolute S convergence < 0.01": eps_s < 0.01,
        "symmetric conversion within empirical tolerance": baseline_conversion
        <= 3 * eps_s,
        "asymmetric conversion change resolved": conversion_change > 3 * eps_s,
        "gap event timing resolved": separation_ok
        and np.isfinite(arrival)
        and abs(arrival - predicted) <= resolution_s,
        "moved gap shift resolved": separation_ok
        and np.isfinite(moved_arrival)
        and abs((moved_arrival - arrival) - expected_shift) <= resolution_s,
    }
    show(pd.Series(checks, name="Passed").to_frame())
    print(f"Empirical eps_S={eps_s:.3g}; eps_rho={eps_rho:.3g}")
    print(
        f"Gap arrival={arrival * 1e12:.2f} ps; moved arrival={moved_arrival * 1e12:.2f} ps"
    )
    OUTPUT.mkdir(parents=True, exist_ok=True)
    convergence.to_csv(OUTPUT / "convergence.csv", index=False)
    (OUTPUT / "validation.json").write_text(
        json_text(
            {
                "checks": {k: bool(v) for k, v in checks.items()},
                "eps_s": float(eps_s),
                "eps_rho": float(eps_rho),
                "arrival_s": float(arrival),
                "moved_arrival_s": float(moved_arrival),
            }
        )
    )
    print(
        "All EM criteria passed."
        if all(checks.values())
        else "EM validation INCONCLUSIVE/FAILED: inspect refinement and time resolution."
    )
else:
    print(
        "EM convergence validation INCOMPLETE. Enable RUN_CONVERGENCE and complete the runs."
    )

## 12. Executed experiments: what the evidence supports

The local run used Palace **v0.18.0**. Two dense first-order mesh levels (12/150 µm
and 8/100 µm refined/background sizes) and nine direct frequency anchors were run
for the intact and disrupted return planes. The 6 µm second-order direct run was
stopped when host memory reached approximately 55 GiB and swapping began. The
initial iterative run was stopped after benchmarking a faster direct solver.
Neither stopped run supplies a completed network.

A second-order spot check on the **same 12 µm mesh** produced one complete
excitation column at 25 GHz before that study was stopped. It is adverse evidence
about spatial discretization, not a complete four-port validation. The remaining
second-order frequencies/excitations and the proposed airbox/substrate sensitivity
suite were not completed. Thus the full issue acceptance criteria are **not met**.

The tables below distinguish complete matrices, adaptive training convergence,
linear residual warnings, direct-sweep agreement, and mesh/order sensitivity. The
moved-slot experiment is an empirical timing check on the coarse model. Matching
its expected delay cannot establish continuum accuracy when discretization checks
fail.

The nine direct-check frequencies are not all on the 250 MHz output grid.
Both linear and cubic interpolation comparisons are reported: the former includes
a visible interpolation error of its own. These are checks at selected frequencies,
not rigorous error bounds over the full band; the endpoints are training points.

In [ ]:
from scipy.interpolate import CubicSpline

from gsim.palace.validation import validate_solution

RECORDED_LABELS = {
    "coarse intact": "pilot-mumps-intact",
    "coarse disrupted": "pilot-mumps-disrupted",
    "fine intact": "refine8-intact",
    "fine disrupted": "refine8-disrupted",
    "direct intact": "direct12-intact",
    "direct disrupted": "direct12-disrupted",
    "moved slot": "moved12",
}
recorded = {key: load_recorded_case(label) for key, label in RECORDED_LABELS.items()}
quality_rows = []
for name, (net, manifest, directory) in recorded.items():
    if net is None:
        quality_rows.append({"case": name, "data": "INCOMPLETE"})
        continue
    config = json.loads((directory / "config.json").read_text())
    log_file = directory / "solver.log"
    report = validate_solution(
        log_file.read_text() if log_file.exists() else "",
        estimator_disabled=config["Solver"]["Linear"].get("EstimatorMaxIts") == 0,
        expected_excitations=[1, 2, 3, 4]
        if manifest["spec"]["settings"]["adaptive_tol"] > 0
        else [],
        exit_code=0,
    )
    stats = json.loads((directory / "output/palace/palace.json").read_text())["Problem"]
    quality_rows.append(
        {
            "case": name,
            "data": "complete",
            "frequencies": len(net.f),
            "field order": config["Solver"]["Order"],
            "unknowns": stats["DegreesOfFreedom"],
            "linear checks": report.core.status,
            "adaptive checks": ", ".join(a.check.status for a in report.adaptive)
            or "direct",
            "adaptive samples": ", ".join(str(a.sample_count) for a in report.adaptive),
            "max reciprocity residual": np.abs(net.s - net.s.transpose(0, 2, 1)).max(),
            "max singular value": np.linalg.svd(net.s, compute_uv=False).max(),
        }
    )
quality = pd.DataFrame(quality_rows)
show(quality)
if OUTPUT.exists():
    quality.to_csv(OUTPUT / "recorded-run-quality.csv", index=False)

required = [
    "coarse intact",
    "coarse disrupted",
    "fine intact",
    "fine disrupted",
    "direct intact",
    "direct disrupted",
]
OBSERVED_READY = all(recorded[name][0] is not None for name in required)
if OBSERVED_READY:
    mm_recorded = {name: mixed_mode(recorded[name][0])[1] for name in required}
    comparisons = []
    for geometry in ("intact", "disrupted"):
        coarse = recorded["coarse " + geometry][0]
        fine = recorded["fine " + geometry][0]
        direct = recorded["direct " + geometry][0]
        np.testing.assert_allclose(coarse.f, fine.f)
        interp = interpolate_complex(coarse, direct.f)
        cubic = CubicSpline(coarse.f, coarse.s, axis=0)(direct.f)
        differences = np.abs(fine.s - coarse.s)
        comparisons.append(
            {
                "geometry": geometry,
                "max direct vs adaptive complex S": np.abs(interp - direct.s).max(),
                "max direct vs adaptive (cubic)": np.abs(cubic - direct.s).max(),
                "max mesh change, all frequencies": differences.max(),
                "max mixed-mode mesh change": np.abs(
                    mm_recorded["fine " + geometry].s
                    - mm_recorded["coarse " + geometry].s
                ).max(),
                "max mesh change, <=50 GHz": differences[fine.f <= 50e9].max(),
                "max mesh change, <=100 GHz": differences[fine.f <= 100e9].max(),
            }
        )
    show(pd.DataFrame(comparisons))
    pd.DataFrame(comparisons).to_csv(
        OUTPUT / "recorded-mesh-comparison.csv", index=False
    )
    rows = []
    for frequency in (25e9, 50e9, 100e9, 150e9, 200e9):
        for geometry in ("intact", "disrupted"):
            net = recorded["fine " + geometry][0]
            mm = mm_recorded["fine " + geometry]
            i = np.argmin(np.abs(net.f - frequency))
            rows.append(
                {
                    "geometry": geometry,
                    "GHz": net.f[i] / 1e9,
                    "Sdd21 dB": float(db(mm.s[i, 1, 0])),
                    "Sdd11 dB": float(db(mm.s[i, 0, 0])),
                    "NEXT dB": float(db(net.s[i, 1, 0])),
                    "FEXT dB": float(db(net.s[i, 3, 0])),
                    "max conversion amplitude": max(
                        np.abs(mm.s[i, :2, 2:]).max(), np.abs(mm.s[i, 2:, :2]).max()
                    ),
                }
            )
    show(pd.DataFrame(rows))
    pd.DataFrame(rows).to_csv(OUTPUT / "recorded-frequency-metrics.csv", index=False)
    numerical_change = max(row["max mixed-mode mesh change"] for row in comparisons)
    fine_b, fine_g = mm_recorded["fine intact"], mm_recorded["fine disrupted"]
    delta = fine_g.s - fine_b.s
    gap_conversion = max(np.abs(delta[:, :2, 2:]).max(), np.abs(delta[:, 2:, :2]).max())
    print(
        f"Largest mesh change: {numerical_change:.6g}; gap-induced conversion change: {gap_conversion:.6g}"
    )
    print("Convergence criterion |delta S| < 0.01:", numerical_change < 0.01)
    print(
        "Gap conversion exceeds 3 x mesh-change indicator:",
        gap_conversion > 3 * numerical_change,
    )
    # Compare TDR differences across mesh levels using the same frequency grid/window.
    fig, ax = plt.subplots(figsize=(10, 4))
    curves = {}
    for level in ("coarse", "fine"):
        b, g = mm_recorded[level + " intact"], mm_recorded[level + " disrupted"]
        t, rb, _ = tdr(b.f, b.s[:, 0, 0], 2 * Z_SE)
        _, rg, _ = tdr(g.f, g.s[:, 0, 0], 2 * Z_SE)
        curves[level] = (rb, rg)
        ax.plot(t * 1e12, rg - rb, label=level + ": disrupted minus intact")
    tau_fine = differential_delay(fine_b)
    predicted = 2 * tau_fine * GEO.gap_x / GEO.length
    ax.axvline(predicted * 1e12, color="k", ls=":", label="Estimated slot round trip")
    ax.set(xlim=(-10, 120), xlabel="Time (ps)", ylabel="Change in reflected step")
    ax.grid()
    ax.legend()
    fig.tight_layout()
    rho_mesh = max(
        np.abs(curves["fine"][i] - curves["coarse"][i]).max() for i in (0, 1)
    )
    print(
        f"One-way delay: {tau_fine * 1e12:.3f} ps; predicted slot echo: {predicted * 1e12:.3f} ps"
    )
    print(f"Maximum TDR step change across meshes: {rho_mesh:.6g}")
    moved_net = recorded["moved slot"][0]
    if moved_net is not None:
        cb, cg = mm_recorded["coarse intact"], mm_recorded["coarse disrupted"]
        cm = mixed_mode(moved_net)[1]
        tau_coarse = differential_delay(cb)
        # First report the raw timing estimator, then apply the mesh uncertainty gate.
        a0 = gap_arrival(cb, cg, tau_coarse, 2 / BASE.fmax, 1e-4)
        a1 = gap_arrival(cb, cm, tau_coarse, 2 / BASE.fmax, 1e-4)
        expected_shift = 2 * tau_coarse * 1200 / GEO.length
        robust = gap_arrival(cb, cg, tau_coarse, 2 / BASE.fmax, 3 * rho_mesh)
        print(f"Raw gap/moved-gap arrivals: {a0 * 1e12:.3f}, {a1 * 1e12:.3f} ps")
        print(
            f"Observed shift: {(a1 - a0) * 1e12:.3f} ps; expected: {expected_shift * 1e12:.3f} ps"
        )
        print("Gap detected above 3 x mesh TDR variation:", bool(np.isfinite(robust)))
        fig, ax = plt.subplots(figsize=(10, 4))
        t, rb, _ = tdr(cb.f, cb.s[:, 0, 0], 2 * Z_SE)
        for title, net in [("slot at 2400 um", cg), ("slot at 3600 um", cm)]:
            _, r, _ = tdr(net.f, net.s[:, 0, 0], 2 * Z_SE)
            ax.plot(t * 1e12, r - rb, label=title)
        ax.set(
            xlim=(-10, 120),
            xlabel="Time (ps)",
            ylabel="Change from intact reflected step",
        )
        ax.grid()
        ax.legend()
        fig.tight_layout()
else:
    print("Recorded comparison INCOMPLETE: missing one or more full matrices.")

if OBSERVED_READY:
    (OUTPUT / "recorded-analysis.json").write_text(
        json_text(
            {
                "status": "spatial_convergence_failed"
                if numerical_change >= 0.01
                else "full_validation_incomplete",
                "full_convergence_suite_complete": False,
                "max_mixed_mode_mesh_change": float(numerical_change),
                "s_convergence_threshold": 0.01,
                "gap_conversion_change": float(gap_conversion),
                "max_tdr_mesh_change": float(rho_mesh),
                "fine_one_way_delay_s": float(tau_fine),
                "coarse_raw_slot_shift_s": float(a1 - a0)
                if moved_net is not None
                else None,
                "coarse_expected_slot_shift_s": float(expected_shift)
                if moved_net is not None
                else None,
                "timing_feature_above_three_times_mesh_indicator": bool(
                    np.isfinite(robust)
                )
                if moved_net is not None
                else None,
                "note": "Mesh differences are empirical indicators, not error bounds. Second-order check is one excitation only.",
            }
        )
    )
    print("\nINTERPRETATION: Completed EM data; spatial convergence FAILED.")
    print("Adaptive/direct agreement and reciprocity do not establish mesh accuracy.")
    print("Treat crosstalk, conversion, and apparent impedance as exploratory values.")

### Separate low-frequency linear-algebra and higher-order checks

The original one-correction MUMPS solves sometimes missed the requested $10^{-8}$
relative residual tolerance at 0.25 GHz. Repeating that four-port point with up to
five corrections passed and changed the S-matrix by only about $10^{-10}$. This
explains why the linear warning has negligible impact here; the original logs
remain flagged rather than being rewritten as successful.

The partial second-order run below contributes only its independently solved
**first excitation column at 25 GHz**. All other entries in that unfinished run
are excluded. The comparison demonstrates a failed discretization check; it does
not certify the second-order answer as ground truth.

In [ ]:
correction_dir = OUTPUT / "low-frequency-correction-intact"
if (correction_dir / "network.npz").exists() and recorded["coarse intact"][
    0
] is not None:
    with np.load(correction_dir / "network.npz", allow_pickle=False) as saved:
        correction = saved["s"]
    original = recorded["coarse intact"][0].s[:1]
    report = validate_solution(
        (correction_dir / "solver.log").read_text(),
        estimator_disabled=True,
        exit_code=0,
    )
    print("Corrected low-frequency linear check:", report.core.status)
    print("max |S_corrected - S_original|:", np.abs(correction - original).max())

spot_dirs = list(OUTPUT.glob("order2-anchors-intact-*/output/palace/port-S.csv"))
if len(spot_dirs) == 1 and recorded["coarse intact"][0] is not None:
    frame = pd.read_csv(spot_dirs[0])
    frame.columns = [column.strip() for column in frame.columns]
    row = frame.loc[np.isclose(frame.iloc[:, 0], 25.0)].iloc[0]
    coarse = recorded["coarse intact"][0]
    i = np.argmin(np.abs(coarse.f - 25e9))
    spot_rows = []
    # Native solver order p1,p3,p2,p4; explicitly map responses to tutorial names.
    for native, name in enumerate(("p1", "p3", "p2", "p4"), start=1):
        magnitude = float(row[f"|S[{native}][1]| (dB)"])
        phase = float(row[f"arg(S[{native}][1]) (deg.)"])
        value = 10 ** (magnitude / 20) * np.exp(1j * np.deg2rad(phase))
        old = coarse.s[i, coarse.port_names.index(name), 0]
        spot_rows.append(
            {
                "response to p1": name,
                "GHz": 25,
                "first-order dB": float(db(old)),
                "second-order dB": magnitude,
                "complex change": abs(value - old),
            }
        )
    show(pd.DataFrame(spot_rows))
    print(
        "Partial check only: remaining second-order excitation columns were NOT completed."
    )

### Reproducing the recorded profiles

The original full-resolution profile remains in `FULL_SETTINGS`; it is not the
resource-bounded profile analyzed above. To launch fresh resource-bounded cases,
set `EXECUTION = "local"` (or `"cloud"`), select the runtime, and call the existing
`run_case` helper with these settings. Future runs use five correction iterations
and record their solver checks; their manifests will differ from the original
one-correction archives.

```python
coarse = replace(FULL_SETTINGS, order=1, refined_um=12, max_um=150)
finer = replace(FULL_SETTINGS, order=1, refined_um=8, max_um=100)
anchors = replace(coarse, points=9, adaptive_tol=0)
for profile, settings in [("coarse", coarse), ("finer", finer), ("anchors", anchors)]:
    for label, gap in [("intact", False), ("disrupted", True)]:
        run_case(f"repeat-{profile}-{label}", GEO, gap, settings)
run_case("repeat-moved", replace(GEO, gap_x=GEO.gap_x + 1200), True, coarse)
```

Keep all emitted manifests, meshes, port information, configurations, and logs.
A fresh run with a changed parameter is a new experiment, not a replacement for
an unfavorable convergence result.


## 14. Resolving spatial convergence

The first-order dense spectra above are retained as an unsuccessful convergence
study. Before repeating them, compare **complete four-port direct solves at
25, 112.5, and 200 GHz**. These frequencies are exactly present in the original
dense grid, so comparison does not require interpolation.

The recovery study compares second-order elements on coarser meshes with
first-order elements on finer meshes. Increasing field order and refining the
mesh are independent controls. A coarser second-order mesh may represent smooth
fields more efficiently, while the conductor edges still require geometric
resolution. Each row reports the actual solved unknown count and residual checks.
A completed run is not automatically a converged discretization.

The original issue requires all of the following:

| Requirement from #274 | Evidence required |
|---|---|
| IHP device example; PDK-independent simulation API | Pinned IHP setup plus generic geometry/stack/port functions and the synthetic PDK setup check |
| Configurable return-path gap; intact/disrupted comparison | Both geometries solved with the same materials, ports, frequency grid, and numerical profile |
| Sdd/Sdc/Scd/Scc, NEXT/FEXT, and TDR | Full matrices; explicitly named port pairs, impedances, and reference planes; plots of each quantity |
| TDR bandwidth, window, and DC policy | Uniform frequency grid, stated finite bandwidth, window/DC sensitivity, and analytic reconstruction checks |
| Mixed-mode conversion and inverse, including reordered ports | Independent matrix formula, scikit-rf conversion, inverse round trip, and permutations |
| Negligible symmetric conversion within mesh tolerance | Converging baseline, symmetry diagnostic, and changes in the conversion blocks across independently refined discretizations |
| Resolved asymmetric-gap change | Gap-induced complex conversion changes compared with numerical changes for the same observable |
| Analytic impedance/delay and gap location | Analytic checks, resolved EM echo, and a measured location shift consistent with round-trip propagation |

We retain an absolute **0.01 complex-S convergence target** for the network.
The numerical variation in the **conversion blocks** and reflected step is also
reported separately, because an error in through-wave phase and an error in mode
conversion are different observables. These variations are empirical convergence
indicators, not rigorous error bounds. Passing only a symmetry diagnostic cannot
establish accuracy of the entire network.


A real-only MUMPS preconditioner is an optional memory reduction strategy:
`real_preconditioner=True` factors the real part while GMRES still solves the
full complex Maxwell system. It requires more Krylov iterations. The 25 GHz
probe on the original second-order mesh agreed with the full complex
factorization to $2.7\times10^{-8}$ in the solved S-parameter column. Use a
strict complex residual tolerance and verify the resulting networks; changing a
preconditioner does not improve the spatial approximation by itself.


The completed 24 µm second-order anchor run passes its linear checks, but its
largest symmetric cross-mode amplitude at the anchors is **0.02951**. Thus this
coarser second-order discretization is not yet a sufficient reference. Finer
second-order and first-order comparisons, plus higher-order cross-checks, are
tracked below. Set `RUN_RECOVERY=True`, select `RECOVERY_TO_RUN`, and use a new
`RECOVERY_PREFIX` with local/cloud execution to reproduce these experiments
without relying on external helper scripts. The listed profiles use the same
geometry and material assumptions as the original study.


Long runs can be interrupted after some independent frequency/excitation solves
have converged. Such columns are preserved, but a network is marked complete
only after all 16 responses exist at every requested frequency. Recovery logs
identify the retained columns and the additional solves. The finer 4 µm
first-order profile uses the tested real preconditioner with a strict complex
GMRES residual tolerance to limit factorization memory.


### Why a small transverse error matters along a long line

For a roughly quasi-TEM line, the per-length inductance $L'$ and capacitance $C'$
give $\tau\approx\ell\sqrt{L'C'}$ and $Z_c\approx\sqrt{L'/C'}$. A small delay
error changes a through wave approximately by

$$|\Delta S|\approx |S|\,\omega |\Delta\tau|.$$

At 200 GHz and a 40 ps transit, a 0.1% delay error produces about 0.05 radians
of phase error, or $|\Delta S|\approx0.025$ for a through amplitude of 0.5.
That exceeds our 0.01 target even though the timing error is far below the
10 ps TDR resolution. A stable-looking TDR arrival therefore cannot establish
frequency-domain convergence. Resolving the fields near thin layers and conductor
corners matters as well as resolving the wavelength along the line. These
quasi-TEM estimates motivate the study; they are not error bounds for the lossy
three-dimensional Maxwell solve.

### Completed recovery results so far

The **4 µm first-order** network has 918,365 unknowns and passes its linear
checks, but the change from the 8 µm result reaches **0.14366** in mixed-mode S
at the direct anchors. Its baseline cross-mode amplitude reaches **0.02002**.
The **24 µm third-order** network has also been completed by recovering every
missing independent solve. Its baseline cross-mode amplitude is **0.01499**,
compared with **0.02951** for second order at the same nominal mesh sizes.
The second-to-third-order mixed-mode S change is **0.14277**. These results
still fail the network convergence target; neither is designated ground truth.


The completed **12 µm second-order** reference also passes its linear checks.
Its baseline cross-mode amplitude reaches **0.01620**, and its difference from
the 24 µm third-order network reaches **0.07240** in mixed-mode S at the anchors.
Thus the independent finer-mesh/higher-order comparison also fails the 0.01 target.
The recovery study has not yet established a usable reference discretization.


The completed **18 µm second-order** network has 1,022,508 unknowns and a
baseline cross-mode amplitude of **0.01988** at the anchors. The mixed-mode
change from 24 to 18 µm is **0.07502**, and from 18 to 12 µm is **0.11663**.
Every retained column passes its linear checks; this refinement sequence also
fails the spatial target. The next mesh experiments align prisms with the long
traces while refining conductor corners independently. Partial pilot columns
are exploratory until complete networks and independent refinements pass.


In [ ]:
# Fresh anchor experiments are sequential and can be memory intensive.
RUN_RECOVERY = False
RECOVERY_PREFIX = ""  # Use "repeat-" to keep fresh experiments separate.
ANCHOR_SETTINGS = replace(FULL_SETTINGS, fmin=25e9, points=3, adaptive_tol=0)
REAL_PC_SETTINGS = replace(
    ANCHOR_SETTINGS,
    real_preconditioner=True,
    linear_max_iterations=200,
    linear_tolerance=1e-9,
)
RECOVERY_SETTINGS = {
    "convergence-p2h24-intact": replace(
        ANCHOR_SETTINGS, order=2, refined_um=24, max_um=180
    ),
    "convergence-p2h18-intact": replace(
        ANCHOR_SETTINGS, order=2, refined_um=18, max_um=150
    ),
    "convergence-p1h4-real-intact": replace(
        REAL_PC_SETTINGS, order=1, refined_um=4, max_um=70
    ),
    "convergence-p2h12-real-intact": replace(
        REAL_PC_SETTINGS, order=2, refined_um=12, max_um=150
    ),
    "convergence-p3h24-real-intact": replace(
        REAL_PC_SETTINGS, order=3, refined_um=24, max_um=180
    ),
    "convergence-p3h18-real-intact": replace(
        REAL_PC_SETTINGS, order=3, refined_um=18, max_um=150
    ),
}
RECOVERY_TO_RUN = [
    "convergence-p2h24-intact",
    "convergence-p2h18-intact",
    "convergence-p1h4-real-intact",
]
if RUN_RECOVERY:
    if EXECUTION not in {"local", "cloud"}:
        raise ValueError("Set EXECUTION to local/cloud before launching recovery runs")
    for label in RECOVERY_TO_RUN:
        run_case(RECOVERY_PREFIX + label, GEO, False, RECOVERY_SETTINGS[label])

RECOVERY_PROFILES = {
    "first order, 12 um": "pilot-mumps-intact",
    "first order, 8 um": "refine8-intact",
    "first order, 4 um": "convergence-p1h4-real-intact",
    "second order, 24 um": "convergence-p2h24-intact",
    "second order, 18 um": "convergence-p2h18-intact",
    "second order, 12 um (real PC)": "convergence-p2h12-real-intact",
    "third order, 24 um (real PC)": "convergence-p3h24-real-intact",
    "third order, 18 um (real PC)": "convergence-p3h18-real-intact",
}
recovery_runs = {
    name: load_recorded_case(
        RECOVERY_PREFIX + label if label.startswith("convergence-") else label
    )
    for name, label in RECOVERY_PROFILES.items()
}


def recovery_physics_signature(manifest, directory):
    """Compare the generated physical problem while excluding numerical settings."""
    spec = manifest["spec"]
    config = json.loads((directory / "config.json").read_text())
    payload = {
        "geometry": spec["geometry_um"],
        "gap": spec["gap"],
        "air_um": spec["settings"]["air_um"],
        "dielectrics": spec["stack"]["dielectrics"],
        "layer_bounds": {
            key: [layer["zmin"], layer["zmax"]]
            for key, layer in spec["stack"]["layers"].items()
        },
        "length_scale": config["Model"]["L0"],
        "materials": config["Domains"]["Materials"],
        "conductivity": config["Boundaries"]["Conductivity"],
        "ports": config["Boundaries"]["LumpedPort"],
        "absorbing": config["Boundaries"]["Absorbing"],
        "port_names": spec["ports"],
        "pairs": spec["pairs"],
        "reference_planes": spec["reference_planes_um"],
    }
    return hashlib.sha256(json_text(payload).encode()).hexdigest()


recovery_reference_physics = None
recovery_rows = []
anchor_frequencies = np.array([25e9, 112.5e9, 200e9])
recovery_at_anchors = {}
for name, (net, manifest, directory) in recovery_runs.items():
    if net is None:
        recovery_rows.append({"profile": name, "status": "incomplete"})
        continue
    physics = recovery_physics_signature(manifest, directory)
    if recovery_reference_physics is None:
        recovery_reference_physics = physics
    if physics != recovery_reference_physics:
        raise ValueError(
            f"Physical model differs for {name}; compare numerical profiles of the same device"
        )
    indices = np.array(
        [np.argmin(np.abs(net.f - frequency)) for frequency in anchor_frequencies]
    )
    np.testing.assert_allclose(net.f[indices], anchor_frequencies, rtol=0, atol=1)
    mm = mixed_mode(net)[1]
    recovery_at_anchors[name] = mm.s[indices]
    report = validate_solution(
        (directory / "solver.log").read_text(), estimator_disabled=True, exit_code=0
    )
    stats = json.loads((directory / "output/palace/palace.json").read_text())["Problem"]
    recovery_rows.append(
        {
            "profile": name,
            "status": "complete",
            "unknowns": stats["DegreesOfFreedom"],
            "linear checks": report.core.status,
            "max cross-mode amplitude at anchors": max(
                np.abs(mm.s[indices, :2, 2:]).max(), np.abs(mm.s[indices, 2:, :2]).max()
            ),
        }
    )
show(pd.DataFrame(recovery_rows))
recovery_comparisons = []
for first, second in [
    ("first order, 12 um", "first order, 8 um"),
    ("first order, 8 um", "first order, 4 um"),
    ("second order, 24 um", "second order, 18 um"),
    ("second order, 24 um", "third order, 24 um (real PC)"),
    ("first order, 4 um", "third order, 24 um (real PC)"),
    ("first order, 4 um", "second order, 18 um"),
    ("second order, 18 um", "second order, 12 um (real PC)"),
    ("second order, 12 um (real PC)", "third order, 24 um (real PC)"),
    ("third order, 24 um (real PC)", "third order, 18 um (real PC)"),
]:
    if first not in recovery_at_anchors or second not in recovery_at_anchors:
        continue
    delta = recovery_at_anchors[first] - recovery_at_anchors[second]
    recovery_comparisons.append(
        {
            "profiles compared": first + " / " + second,
            "max complex mixed-mode S change": np.abs(delta).max(),
            "max conversion-block change": max(
                np.abs(delta[:, :2, 2:]).max(), np.abs(delta[:, 2:, :2]).max()
            ),
            "network target <0.01": bool(np.abs(delta).max() < 0.01),
        }
    )
if recovery_comparisons:
    show(pd.DataFrame(recovery_comparisons))
    pd.DataFrame(recovery_rows).to_csv(OUTPUT / "recovery-profiles.csv", index=False)
    pd.DataFrame(recovery_comparisons).to_csv(
        OUTPUT / "recovery-comparisons.csv", index=False
    )
else:
    print("Independent recovery comparisons pending complete four-port solves.")

## 16. Align the mesh with the interconnect

The recovery studies show that making every tetrahedron smaller is expensive,
and that the first-order field approximation is inadequate. The next experiment
uses a triangular cross-section swept into prisms along the long traces. This
lets us choose **axial spacing**, **conductor-corner spacing**, and **far-region
spacing** separately. These controls change the finite-element approximation;
all layer heights, dielectric/conductor coefficients, terminal sheets, and outer
boundaries retain their declared physical meaning. Section 15 verified prism
handling for a uniform TEM field. The real conductor corners still need their
own refinement study.

The geometry here has rectangular conductors and disjoint, flat dielectric
intervals. The helper below is specialized to that geometry, and takes its
material names, layer roles, and heights from a supplied stack. It does not use
IHP layer/material names. `MeshResult`, `PalacePort`, and `write_config` connect
this mesh to gsim's ordinary material/port configuration code. The IHP selection
and overrides remain caller inputs. The laboratory uses a local Palace runtime.

Three details matter:

1. **Conductor interiors.** A surface-impedance boundary models the exterior
   fields without resolving fields inside the metal. The original mesher leaves
   dielectric-filled, closed conductor interiors behind those boundaries. Our
   pilot removed these unexcited enclosed regions, retaining exactly the same
   conductor surfaces and coefficients. Two independently solved columns (25
   and 200 GHz, source p1) changed by less than $1.4\times10^{-11}$. This is an
   empirical equivalence check at those points, not a general theorem about all
   meshes. `exterior_only=False` retains the filler for further checks. Do not
   change Palace's `External` flag: that would change its effective-thickness
   convention and the physical boundary model.
2. **Symmetry.** Optional mirroring constructs a reflection-symmetric *mesh*.
   Every requested source is still solved independently on the complete domain.
   No S-parameter matrix is symmetrized. Exact mesh symmetry can suppress
   baseline asymmetry, but cannot establish propagation or gap accuracy.
3. **Paired grids.** Reuse the same cross-section for intact/disrupted/moved-slot
   cases. Include both slot positions in the axial landmarks. The physical hole
   activates dielectric cells previously enclosed by the return metal; geometry
   changes are confined to the intended slot. Refining corners in this cross-
   section does not refine the two axial slot ends. The axial stations must be
   refined separately before accepting a three-dimensional gap response.

**Pilot evidence, still incomplete:** the aligned second-order source-p1 column
changes by **0.00446 at 25 GHz** and **0.01030 at 200 GHz** between the original
100 µm axial / 2 µm corner grid and the 75 µm axial / 0.25 µm corner grid. Both
solves pass the linear checks. These are only two columns of different meshes,
not full-network convergence. The completed 0.125 µm corner study changes that column by **0.00097 at 25 GHz** and **0.00383 at 200 GHz**; all its linear solves pass. Independent axial and order checks are now running.
The 0.01 full mixed-mode target and resolved-conversion/TDR gates remain unchanged.

The code includes controls to run complete direct-anchor or dense four-port
networks, with manifests and hashes. Leave `RUN_ALIGNED=False` to inspect the
implementation without launching a simulation. The listed meshes are candidates,
not accepted references. Run the independent refinement comparisons before
using their gap spectra quantitatively.


**PDK separation check:** `CHECK_ALIGNED_SETUP=True` builds the synthetic
stack below with renamed metals/dielectrics, different heights, and one finite
dielectric region. This checks the new mesh/configuration path. It is a setup
fixture, not another device-validation example. The recorded check passed,
including a Palace configuration dry run. The new IHP mesh/configuration also
matches the prototype's nodes, element roles, and physical coefficients exactly
(up to physical-group identifiers and internal-interface bookkeeping).


`axial_end_um` adds geometric grading on both sides of terminal and slot-end
planes over `axial_end_extent_um`. For example, compare 2, 1, and 0.5 µm at
fixed transverse mesh, then refine the transverse mesh independently. The pilot
columns above used **no axial end grading**; the graded candidate has not been
validated. Reducing the nominal axial wavelength spacing alone can leave a
slot's first two cells almost unchanged, so that comparison would be weak
evidence about its localized reflection. The same graded landmarks are used in
every paired geometry.


**Dense-sweep solver check:** direct anchors can miss a nearly singular
unshifted real preconditioner. A small IHP pipeline fixture hit repeated GMRES
failures near 10.8 GHz with that choice; its incomplete dense data are rejected.
The laboratory therefore uses `PCMatShifted=True` with the real preconditioner:
the physical complex Maxwell operator remains unshifted. At 25 and 200 GHz the
large pilot's shifted/unshifted independently solved columns agree within
$3.1\times10^{-9}$ after both residual checks pass. Positive shifting requires
more Krylov iterations (460 at 200 GHz in that probe); the iteration limit is
explicit, and any failed main solve still rejects the complete network. The shifted small-fixture full-band check has now completed all 800 frequencies
and all four independent excitations with every linear/adaptive check passing.
This is a 200 µm IHP pipeline fixture, not a substitute for the 6 mm device.
The large-model dense and spatial-convergence checks remain open.


### 16.1 Cross-section mesh

Set the execution and mesh controls here. The cross-section generator fragments
the dielectric intervals and conductor/terminal boundaries before triangulation.
Its optional reflection step mirrors mesh triangles, not simulated responses.

In [ ]:
# Independent axial/corner/field-order candidates. All dimensions are µm.
RUN_ALIGNED = False
ALIGNED_THREADS = 1
ALIGNED_BLAS_THREADS = (
    4  # 4 MPI processes x 4 BLAS threads on the recorded 16-core host
)
ALIGNED_MPI_ARGS = "--bind-to none"
ALIGNED_DENSE = False  # True: 800 points, 0.25-200 GHz; False: direct anchors
ALIGNED_PROFILE = {
    "order": 2,
    "axial_um": 75.0,
    "corner_um": 0.125,
    "far_um": 40.0,
    "air_um": 150.0,
    "mirror": True,
    "exterior_only": True,
    "axial_end_um": 1.0,
    "axial_end_extent_um": 20.0,
    "positive_preconditioner": True,
    "linear_max_iterations": 1000,
    "linear_restart": 120,
}
ALIGNED_ANCHORS_GHZ = (25.0, 112.5, 200.0)
ALIGNED_SLOT_CENTRES = (GEO.gap_x, GEO.gap_x + 1200.0)


import itertools
import json
from pathlib import Path

import gmsh
import numpy as np


def aligned_cross_section(
    g,
    signal_z,
    return_z,
    dielectrics,
    margin,
    corner,
    hfar,
    interior_alignment=True,
    point_sizing=True,
    mirror=False,
):
    W, w, s = g["ground_width"], g["width"], g["separation"]
    gy, gw = g["gap_y"], g["gap_width"]
    zmin = min(d["zmin"] for d in dielectrics)
    zmax = max(d["zmax"] for d in dielectrics)
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("cross_section")
    occ = gmsh.model.occ
    rects = [
        occ.addRectangle(
            -W / 2 - margin, zmin - margin, 0, W + 2 * margin, zmax - zmin + 2 * margin
        )
    ]
    for d in dielectrics:
        rects.append(occ.addRectangle(-W / 2, d["zmin"], 0, W, d["zmax"] - d["zmin"]))  # noqa: PERF401 -- explicit CAD entities
    cuts = np.unique(
        [
            -W / 2,
            W / 2,
            gy - gw / 2,
            gy + gw / 2,
            -gy - gw / 2,
            -gy + gw / 2,
            *[c + q for c in [-s / 2, s / 2] for q in [-w / 2, w / 2]],
        ]
    )
    for a, b in itertools.pairwise(cuts):
        rects.append(
            occ.addRectangle(a, return_z[0], 0, b - a, return_z[1] - return_z[0])
        )
    for center in [-s / 2, s / 2]:
        rects.append(
            occ.addRectangle(
                center - w / 2, signal_z[0], 0, w, signal_z[1] - signal_z[0]
            )
        )
        rects.append(
            occ.addRectangle(
                center - w / 2, return_z[1], 0, w, signal_z[0] - return_z[1]
            )
        )
    for center in [gy, -gy]:
        rects.append(  # noqa: PERF401 -- explicit CAD entities
            occ.addRectangle(
                center - gw / 2, return_z[0], 0, gw, return_z[1] - return_z[0]
            )
        )
    if mirror:
        rects.append(
            occ.addRectangle(
                0, zmin - margin, 0, W / 2 + margin, zmax - zmin + 2 * margin
            )
        )
    occ.fragment([(2, rects[0])], [(2, t) for t in rects[1:]])
    occ.synchronize()
    # Small edges/corners near conductors; graded outer air/substrate.
    gmsh.option.setNumber("Mesh.MeshSizeMin", corner * 0.25)
    gmsh.option.setNumber("Mesh.MeshSizeMax", hfar)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", int(point_sizing))
    gmsh.option.setNumber("Mesh.Algorithm", 6)
    gmsh.option.setNumber("General.NumThreads", 1)
    gmsh.model.mesh.setSize(gmsh.model.getEntities(0), hfar)
    near = []
    for dim, tag in gmsh.model.getEntities(0):
        y, z, _ = gmsh.model.getValue(dim, tag, [])
        if abs(y) <= W / 2 + 1e-8 and return_z[0] - 1e-8 <= z <= signal_z[1] + 1e-8:
            near.append((dim, tag))
    gmsh.model.mesh.setSize(near, corner)
    distance = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(distance, "PointsList", [tag for _, tag in near])
    threshold = gmsh.model.mesh.field.add("Threshold")
    for key, value in [
        ("InField", distance),
        ("SizeMin", corner),
        ("SizeMax", hfar),
        ("DistMin", corner),
        ("DistMax", 16.0),
    ]:
        gmsh.model.mesh.field.setNumber(threshold, key, value)
    gmsh.model.mesh.field.setAsBackgroundMesh(threshold)
    # Align triangles within thin rectangular conductor interiors, instead of
    # forcing isotropic elements along a 0.42 um return layer.
    for dim, tag in gmsh.model.getEntities(2):
        xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
        thin = abs(ymin - return_z[0]) < 1e-6 and abs(ymax - return_z[1]) < 1e-6
        trace = (
            abs(ymin - signal_z[0]) < 1e-6
            and abs(ymax - signal_z[1]) < 1e-6
            and xmax - xmin <= w + 1e-6
        )
        if not interior_alignment or not (thin or trace):
            continue
        boundary = gmsh.model.getBoundary([(dim, tag)], oriented=False)
        if len(boundary) != 4:
            continue
        for _, edge in boundary:
            x0, y0, _, x1, y1, _ = gmsh.model.getBoundingBox(1, edge)
            length = max(x1 - x0, y1 - y0)
            target = 2 * corner if x1 - x0 > y1 - y0 else corner
            gmsh.model.mesh.setTransfiniteCurve(
                edge, max(2, int(np.ceil(length / target - 1e-7)) + 1)
            )
        gmsh.model.mesh.setTransfiniteSurface(tag)
    gmsh.model.mesh.generate(2)
    tags, coords, _ = gmsh.model.mesh.getNodes()
    coords = np.asarray(coords).reshape(-1, 3)
    order = np.argsort(tags)
    tags = np.asarray(tags)[order]
    yz = coords[order, :2]
    types, _, node_lists = gmsh.model.mesh.getElements(2)
    triangles = []
    for typ, nodes in zip(types, node_lists, strict=True):
        if typ != 2:
            raise ValueError("Expected linear triangles")
        triangles.extend(np.searchsorted(tags, np.asarray(nodes).reshape(-1, 3)))
    tri = np.asarray(triangles, dtype=np.int64)
    points = yz[tri]
    a = points[:, 1] - points[:, 0]
    b = points[:, 2] - points[:, 0]
    cross = a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0]
    assert (cross > 0).all()
    gmsh.finalize()
    if mirror:
        positive = tri[np.min(yz[tri, 0], axis=1) >= -1e-9]
        used = np.unique(positive)
        points = yz[used].copy()
        points[np.abs(points[:, 0]) < 1e-9, 0] = 0.0
        triangles = np.searchsorted(used, positive)
        offaxis = np.flatnonzero(points[:, 0] > 0.0)
        reflection = np.arange(len(points))
        reflection[offaxis] = np.arange(len(points), len(points) + len(offaxis))
        reflected = points[offaxis].copy()
        reflected[:, 0] *= -1
        yz = np.concatenate([points, reflected])
        tri = np.concatenate([triangles, reflection[triangles[:, [0, 2, 1]]]])
        q = yz[tri]
        det = (q[:, 1, 0] - q[:, 0, 0]) * (q[:, 2, 1] - q[:, 0, 1]) - (
            q[:, 1, 1] - q[:, 0, 1]
        ) * (q[:, 2, 0] - q[:, 0, 0])
        assert np.min(det) > 0
        expected = (W + 2 * margin) * (zmax - zmin + 2 * margin)
        np.testing.assert_allclose(np.sum(det) / 2, expected, rtol=1e-10)
    return yz, tri

### 16.2 Axial stations and prism assembly

The station builder resolves terminal and slot-end planes, with optional local
grading. Sweeping the section gives a conforming prism mesh. Physical groups
identify dielectric regions, conductor surfaces, ports, and the outer boundary.

In [ ]:
def aligned_axial_stations(
    length,
    margin,
    gap_length,
    slot_centres,
    max_step,
    end_step=None,
    end_extent=20.0,
    growth=2.5,
):
    """Refine actual conductor/slot ends; use identical landmarks in paired cases."""
    if min(length, margin, gap_length, max_step) <= 0:
        raise ValueError("Use positive dimensions and axial spacing")
    features = np.unique(
        [
            0.0,
            length,
            *[x + q for x in slot_centres for q in (-gap_length / 2, gap_length / 2)],
        ]
    )
    if np.any(features < 0) or np.any(features > length):
        raise ValueError("Slot landmarks must remain inside the line")
    knots = [-margin, length + margin, *features]
    if end_step is not None:
        if end_step <= 0 or end_extent <= 0 or growth <= 1:
            raise ValueError(
                "Use positive end spacing/extent and growth greater than one"
            )
        neighbours = np.r_[-margin, features, length + margin]
        for i, feature in enumerate(features, 1):
            for direction, neighbour in [
                (-1, neighbours[i - 1]),
                (1, neighbours[i + 1]),
            ]:
                limit = min(end_extent, abs(feature - neighbour) / 2)
                distance, step = 0.0, min(end_step, max_step)
                while distance + step < limit:
                    distance += step
                    knots.append(feature + direction * distance)
                    step = min(growth * step, max_step)
                if limit > 0:
                    knots.append(feature + direction * limit)
    knots = np.unique(np.round(knots, 12))
    xs = np.concatenate(
        [
            np.linspace(a, b, int(np.ceil((b - a) / max_step)) + 1)[:-1]
            for a, b in itertools.pairwise(knots)
        ]
        + [knots[-1:]]
    )
    assert np.all(np.diff(xs) > 0)
    assert np.max(np.diff(xs)) <= max_step * (1 + 1e-12)
    return xs


def make_aligned_mesh(
    path,
    g,
    signal_z,
    return_z,
    dielectrics,
    margin=150.0,
    hx=150.0,
    corner=2.0,
    hfar=40.0,
    gap=False,
    section=None,
    axial_stations=None,
    exclude_interiors=False,
    interior_alignment=True,
    point_sizing=True,
    mirror=False,
    signal_key="signal",
    return_key="return",
    slot_centres=None,
    axial_end_um=None,
    axial_end_extent_um=20.0,
):
    yz, tri = (
        aligned_cross_section(
            g,
            signal_z,
            return_z,
            dielectrics,
            margin,
            corner,
            hfar,
            interior_alignment,
            point_sizing,
            mirror,
        )
        if section is None
        else section
    )
    L, W, w, s = g["length"], g["ground_width"], g["width"], g["separation"]
    gx, gl, gy, gw = g["gap_x"], g["gap_length"], g["gap_y"], g["gap_width"]
    centres = (gx,) if slot_centres is None else tuple(slot_centres)
    xs = aligned_axial_stations(
        L,
        margin,
        gl,
        centres,
        hx,
        end_step=axial_end_um,
        end_extent=axial_end_extent_um,
    )
    if axial_stations is not None:
        xs = np.asarray(axial_stations, float)
    for required in [-margin, 0, gx - gl / 2, gx + gl / 2, L, L + margin]:
        if np.min(np.abs(xs - required)) > 1e-8:
            raise ValueError("Axial stations omit a device/slot boundary")
    nx = len(xs) - 1
    n2 = len(yz)
    nt = len(tri)
    Y, Z = yz[tri].mean(axis=1).T
    X = (xs[:-1] + xs[1:]) / 2
    X = X[:, None]
    Y = Y[None, :]
    Z = Z[None, :]
    body = (X > 0) & (X < L) & (np.abs(Y) < W / 2)
    ids = {
        name: i + 1
        for i, name in enumerate(sorted({"air", *[d["material"] for d in dielectrics]}))
    }
    base = len(ids)
    outer = base + 9
    material = np.full((nx, nt), ids["air"], dtype=np.int64)
    for d in dielectrics:
        material[body & (d["zmin"] < Z) & (d["zmax"] > Z)] = ids[d["material"]]
    conductor = np.zeros_like(material)
    ret = body & (return_z[0] < Z) & (return_z[1] > Z)
    if gap:
        ret &= ~((abs(X - gx) < gl / 2) & (abs(Y - gy) < gw / 2))
    conductor[ret] = 1
    conductor[
        (X > 0)
        & (X < L)
        & ((abs(Y - s / 2) < w / 2) | (abs(Y + s / 2) < w / 2))
        & (signal_z[0] < Z)
        & (signal_z[1] > Z)
    ] = 2
    node = np.arange(1, (nx + 1) * n2 + 1).reshape(nx + 1, n2)
    prisms = np.concatenate([node[:-1, tri], node[1:, tri]], axis=-1).reshape(-1, 6)
    coords = np.concatenate([np.c_[np.full(n2, x), yz] for x in xs])
    interfaces = {
        (a, b): base + 10 + i
        for i, (a, b) in enumerate(
            (a, b) for a in ids.values() for b in ids.values() if a < b
        )
    }

    def labels(cl, cr, ml, mr, zface=False):
        a = np.zeros(cl.shape, dtype=np.int64)
        a[(cl == -1) | (cr == -1)] = outer
        for pair, tag in interfaces.items():
            a[
                ((ml == pair[0]) & (mr == pair[1]))
                | ((ml == pair[1]) & (mr == pair[0]))
            ] = tag
        for code, xy, side in [(1, base + 1, base + 2), (2, base + 3, base + 4)]:
            a[(cl != cr) & ((cl == code) | (cr == code))] = xy if zface else side
        return a

    triangles = []
    quads = []
    for i, x in enumerate(xs):
        cl = conductor[i - 1] if i else np.full(nt, -1)
        cr = conductor[i] if i < nx else np.full(nt, -1)
        ml = material[i - 1] if i else np.full(nt, -1)
        mr = material[i] if i < nx else np.full(nt, -1)
        lab = labels(cl, cr, ml, mr)
        if abs(x) < 1e-8 or abs(x - L) < 1e-8:
            for center, near, far in [
                (s / 2, base + 5, base + 6),
                (-s / 2, base + 7, base + 8),
            ]:
                port = (
                    (abs(Y[0] - center) < w / 2)
                    & (Z[0] > return_z[1])
                    & (Z[0] < signal_z[0])
                )
                lab[port] = near if abs(x) < 1e-8 else far
        for j in np.flatnonzero(lab):
            triangles.append((int(lab[j]), node[i, tri[j]].tolist()))  # noqa: PERF401 -- explicit faces
    edges = {}
    for j, t in enumerate(tri):
        for a, b in [(t[0], t[1]), (t[1], t[2]), (t[2], t[0])]:
            edges.setdefault(tuple(sorted((int(a), int(b)))), []).append(j)
    for (a, b), adj in edges.items():
        assert len(adj) <= 2
        left = adj[0]
        right = adj[1] if len(adj) == 2 else None
        cl = conductor[:, left]
        ml = material[:, left]
        cr = conductor[:, right] if right is not None else np.full(nx, -1)
        mr = material[:, right] if right is not None else np.full(nx, -1)
        lab = labels(cl, cr, ml, mr, zface=abs(yz[a, 1] - yz[b, 1]) < 1e-8)
        for i in np.flatnonzero(lab):
            quads.append(  # noqa: PERF401 -- explicit faces
                (int(lab[i]), [node[i, a], node[i, b], node[i + 1, b], node[i + 1, a]])
            )
    volume_labels = material.ravel()
    removed_count = 0
    if exclude_interiors:
        keep = conductor.ravel() == 0
        removed_count = int((~keep).sum())
        prisms = prisms[keep]
        volume_labels = volume_labels[keep]
        used = np.unique(
            np.concatenate(
                [
                    prisms.ravel(),
                    np.asarray(
                        [n for _, vs in triangles + quads for n in vs], dtype=np.int64
                    ),
                ]
            )
        )
        remap = np.zeros(len(coords) + 1, dtype=np.int64)
        remap[used] = np.arange(1, len(used) + 1)
        coords = coords[used - 1]
        prisms = remap[prisms]
        triangles = [(tag, remap[np.asarray(vs)].tolist()) for tag, vs in triangles]
        quads = [(tag, remap[np.asarray(vs)].tolist()) for tag, vs in quads]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    names = {tag: name for name, tag in ids.items()}
    names.update(
        {
            base + 1: f"{return_key}_xy",
            base + 2: f"{return_key}_z",
            base + 3: f"{signal_key}_xy",
            base + 4: f"{signal_key}_z",
            **{base + 5 + i: f"P{i + 1}" for i in range(4)},
            outer: "outer",
        }
    )
    for pair, tag in interfaces.items():
        names[tag] = "interface_" + names[pair[0]] + "_" + names[pair[1]]
    groups = {
        "volumes": {name: {"phys_group": tag} for name, tag in ids.items()},
        "conductor_surfaces": {
            names[tag]: {"phys_group": tag} for tag in range(base + 1, base + 5)
        },
        "port_surfaces": {
            f"P{i + 1}": {"phys_group": base + 5 + i, "type": "via"} for i in range(4)
        },
        "boundary_surfaces": {"absorbing": {"phys_group": outer}},
    }
    with path.open("w") as f:
        f.write(f"$MeshFormat\n2.2 0 8\n$EndMeshFormat\n$PhysicalNames\n{len(names)}\n")
        for tag, name in names.items():
            f.write(f'{3 if tag <= base else 2} {tag} "{name}"\n')
        f.write(f"$EndPhysicalNames\n$Nodes\n{len(coords)}\n")
        for i, (x, y, z) in enumerate(coords, 1):
            f.write(f"{i} {x:.12g} {y:.12g} {z:.12g}\n")
        f.write(f"$EndNodes\n$Elements\n{len(triangles) + len(quads) + len(prisms)}\n")
        eid = 0
        for typ, entries in [
            (2, triangles),
            (3, quads),
            (6, zip(volume_labels, prisms, strict=True)),
        ]:
            for tag, q in entries:
                eid += 1
                f.write(f"{eid} {typ} 2 {tag} {tag} " + " ".join(map(str, q)) + "\n")
        f.write("$EndElements\n")
    return {
        "groups": groups,
        "material_ids": ids,
        "mirror": mirror,
        "axial_end_um": axial_end_um,
        "axial_end_extent_um": axial_end_extent_um,
        "nodes": len(coords),
        "prisms": len(prisms),
        "exterior_only": exclude_interiors,
        "inactive_conductor_cells_removed": removed_count,
        "interior_alignment": interior_alignment,
        "point_sizing": point_sizing,
        "cross_section_nodes": n2,
        "cross_section_triangles": nt,
        "nx": nx,
        "hx_um": hx,
        "corner_um": corner,
        "hfar_um": hfar,
        "gap": gap,
        "mesh_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }

### 16.3 Configure from a supplied layer stack

This adapter obtains conductor heights/materials and dielectric intervals from
its arguments. It writes a `MeshResult` and uses gsim to generate the material
and port coefficients. Advanced linear controls are explicit in the saved JSON.

In [ ]:
import copy
import json
from pathlib import Path

import numpy as np

from gsim.palace.mesh.config_generator import write_config
from gsim.palace.mesh.generator import MeshResult
from gsim.palace.models import DrivenConfig, SolverConfig
from gsim.palace.ports.config import PalacePort, PortGeometry


def aligned_config(
    stack,
    geometry,
    signal_key,
    return_key,
    directory,
    *,
    profile,
    overrides=None,
    gap=False,
    section=None,
    slot_centres=None,
    frequencies_ghz=(25.0, 112.5, 200.0),
    dense=False,
):
    """Geometry and physical coefficients come from caller-supplied stack/roles."""
    if signal_key == return_key or not {signal_key, return_key} <= stack.layers.keys():
        raise ValueError("Select two distinct conductor layers")
    stack = copy.deepcopy(stack)
    for material, properties in (overrides or {}).items():
        stack.materials.setdefault(material, {}).update(properties)
    sig, ret = stack.layers[signal_key], stack.layers[return_key]
    if sig.zmin <= ret.zmax:
        raise ValueError("Signal must lie above return")
    for layer in [sig, ret]:
        if not stack.materials.get(layer.material, {}).get("conductivity", 0) > 0:
            raise ValueError("Conductor conductivity must be explicit")
    ds = sorted(stack.dielectrics, key=lambda d: d["zmin"])
    for a, b in itertools.pairwise(ds):
        if a["zmax"] > b["zmin"] + 1e-9:
            raise ValueError(
                "This aligned mesher requires disjoint flat dielectric intervals"
            )
    if not ds:
        raise ValueError("Supply explicit finite dielectric regions")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    summary = make_aligned_mesh(
        directory / "palace.msh",
        geometry,
        (sig.zmin, sig.zmax),
        (ret.zmin, ret.zmax),
        stack.dielectrics,
        margin=profile["air_um"],
        hx=profile["axial_um"],
        corner=profile["corner_um"],
        hfar=profile["far_um"],
        gap=gap,
        section=section,
        exclude_interiors=profile["exterior_only"],
        interior_alignment=False,
        point_sizing=False,
        mirror=profile["mirror"],
        signal_key=signal_key,
        return_key=return_key,
        slot_centres=slot_centres,
        axial_end_um=profile.get("axial_end_um"),
        axial_end_extent_um=profile.get("axial_end_extent_um", 20.0),
    )
    ports = []
    port_info = []
    L, w, s = geometry["length"], geometry["width"], geometry["separation"]
    for i, (name, x, y) in enumerate(
        [("p1", 0.0, s / 2), ("p3", L, s / 2), ("p2", 0.0, -s / 2), ("p4", L, -s / 2)],
        1,
    ):
        ports.append(
            PalacePort(
                name=name,
                geometry=PortGeometry.INTERLAYER,
                center=(x, y),
                width=w,
                orientation=180.0 if x == 0 else 0.0,
                zmin=ret.zmax,
                zmax=sig.zmin,
                from_layer=return_key,
                to_layer=signal_key,
                impedance=50.0,
                excited=True,
            )
        )
        port_info.append(
            {
                "portnumber": i,
                "name": name,
                "Z0": 50.0,
                "type": "via",
                "direction": "Z",
                "length": sig.zmin - ret.zmax,
                "width": w,
                "xmin": x,
                "xmax": x,
                "ymin": y - w / 2,
                "ymax": y + w / 2,
                "zmin": ret.zmax,
                "zmax": sig.zmin,
            }
        )
    mesh = MeshResult(
        mesh_path=directory / "palace.msh",
        port_info=port_info,
        mesh_stats={"nodes": summary["nodes"], "prisms": summary["prisms"]},
        groups=summary["groups"],
        output_dir=directory,
        model_name="palace",
        fmax=200e9,
    )
    cfgpath = write_config(
        mesh,
        stack,
        ports,
        driven_config=DrivenConfig(
            fmin=0.25e9 if dense else min(frequencies_ghz) * 1e9,
            fmax=200e9 if dense else max(frequencies_ghz) * 1e9,
            num_points=800 if dense else len(frequencies_ghz),
            adaptive_tol=1e-5 if dense else 0.0,
            adaptive_max_samples=80,
        ),
        numerical_config=SolverConfig(order=profile["order"]),
    )
    cfg = json.loads(cfgpath.read_text())
    cfg["Problem"]["Verbose"] = 2
    cfg["Solver"]["Linear"].update(
        Type="MUMPS",
        KSPType="GMRES",
        Tol=1e-9,
        MaxIts=profile.get("linear_max_iterations", 1000),
        MGMaxLevels=1,
        EstimatorMaxIts=0,
        DivFreeMaxIts=0,
        PCMatReal=True,
        ComplexCoarseSolve=False,
        MaxSize=profile.get("linear_restart", 120),
        PCMatShifted=profile.get("positive_preconditioner", True),
    )
    if not dense:
        cfg["Solver"]["Driven"] = {
            "Samples": [{"Type": "Point", "Freq": list(frequencies_ghz)}]
        }
    cfgpath.write_text(json.dumps(cfg, indent=2))
    (directory / "mesh-summary.json").write_text(json.dumps(summary, indent=2))
    return mesh, cfg, summary

### 16.4 Read and preserve complete four-port experiments

Read every independently solved column, apply the declared port permutation,
and reject missing entries or failed linear/adaptive solves. A completed archive
contains the full network, inputs, solver evidence, and hashes.

Reloading rechecks the input hashes, actual solver log and exit, every adaptive
outcome, grid, canonical ports, impedances, and all entries against the native
CSV. A previously green manifest alone is not accepted as evidence.


A main GMRES failure terminates the owned native job immediately and preserves
its partial log as rejected data. This prevents an adaptive sweep from repeatedly
training on unconverged fields. The deliberately disabled estimator's PCG
messages do not trigger this rule. Actual terminal geometry/assignments and
field order are also rechecked when loading an archive.


In [ ]:
import signal
from contextlib import suppress

import psutil


def aligned_read_network(directory):
    frame = pd.read_csv(directory / "output/palace/port-S.csv")
    frame.columns = frame.columns.str.strip()
    s = np.empty((len(frame), 4, 4), complex)
    for source in range(1, 5):
        for response in range(1, 5):
            magnitude = frame[f"|S[{response}][{source}]| (dB)"].to_numpy()
            phase = frame[f"arg(S[{response}][{source}]) (deg.)"].to_numpy()
            s[:, response - 1, source - 1] = 10 ** (magnitude / 20) * np.exp(
                1j * np.deg2rad(phase)
            )
    if not np.isfinite(s).all():
        raise ValueError("Missing/nonfinite independently solved four-port response")
    # Native order p1,p3,p2,p4 -> canonical near A, near B, far A, far B.
    idx = [0, 2, 1, 3]
    net = rf.Network(
        f=frame.iloc[:, 0].to_numpy() * 1e9,
        s=s[:, idx][:, :, idx],
        z0=Z_SE,
        f_unit="Hz",
    )
    net.port_names = list(PORTS)
    net.name = directory.name
    check_network(net)
    return net


def aligned_check_solver(directory, manifest, dense):
    """Re-read actual inputs/logs; an exit code or old green manifest is insufficient."""
    for name, expected in manifest["inputs"].items():
        if hashlib.sha256((directory / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"Changed archived input: {name}")
    config = json.loads((directory / "config.json").read_text())
    validate_generated_config(config)
    spec = manifest["spec"]
    if config["Solver"]["Order"] != spec["profile"]["order"]:
        raise ValueError("Recorded field order disagrees with the actual config")
    length, width, spacing = (
        spec["geometry_um"][key] for key in ("length", "width", "separation")
    )
    if spec["reference_planes_um"] != [0.0, length]:
        raise ValueError("Reference planes differ from the device ends")
    info = json.loads((directory / "port_information.json").read_text())["ports"]
    expected = [
        ("p1", 0.0, spacing / 2),
        ("p3", length, spacing / 2),
        ("p2", 0.0, -spacing / 2),
        ("p4", length, -spacing / 2),
    ]
    if len(info) != 4 or [p["name"] for p in info] != spec["port_names_native"]:
        raise ValueError("Native port names/order disagree with the archive")
    bottom = spec["layers"][spec["return_key"]]["zmax"]
    top = spec["layers"][spec["signal_key"]]["zmin"]
    for index, (port, (name, x, y)) in enumerate(zip(info, expected, strict=True), 1):
        if port["name"] != name or port["Z0"] != spec["reference_ohm"]:
            raise ValueError("Port identity/reference impedance differs")
        np.testing.assert_allclose(
            [port[k] for k in ("xmin", "xmax", "ymin", "ymax", "zmin", "zmax")],
            [x, x, y - width / 2, y + width / 2, bottom, top],
            rtol=0,
            atol=1e-10,
        )
        entry = next(
            p for p in config["Boundaries"]["LumpedPort"] if p["Index"] == index
        )
        attribute = manifest["mesh"]["groups"]["port_surfaces"][f"P{index}"][
            "phys_group"
        ]
        if entry["Attributes"] != [attribute] or entry["Excitation"] != index:
            raise ValueError(
                "Excitation/measurement port assignment differs from the recorded geometry"
            )
    log = (directory / "solver.log").read_bytes()
    if (
        "solver_log_sha256" in manifest
        and hashlib.sha256(log).hexdigest() != manifest["solver_log_sha256"]
    ):
        raise ValueError("Changed archived solver log")
    if manifest.get("exit_code") != 0:
        raise ValueError("A successful recorded Palace exit is required")
    evidence = validate_solution(
        log.decode(),
        estimator_disabled=True,
        expected_excitations=[1, 2, 3, 4] if dense else [],
        exit_code=manifest["exit_code"],
    )
    if evidence.core.status != "passed":
        raise ValueError(f"Main field solve did not pass: {evidence.core.status}")
    if dense and (
        len(evidence.adaptive) != 4
        or any(a.check.status != "passed" for a in evidence.adaptive)
    ):
        raise ValueError("Every independent excitation must pass its adaptive sweep")
    return evidence


def aligned_load_archive(directory, manifest):
    """Validate complete independently solved data, including its declared grid/order."""
    if manifest.get("status") != "complete":
        raise ValueError("Incomplete archive")
    spec = manifest["spec"]
    aligned_check_solver(directory, manifest, spec["dense"])
    data_path = directory / "network.npz"
    if hashlib.sha256(data_path.read_bytes()).hexdigest() != manifest["sha256"]:
        raise ValueError("Changed archived network")
    with np.load(data_path, allow_pickle=False) as saved:
        net = rf.Network(f=saved["f"], s=saved["s"], z0=saved["z0"], f_unit="Hz")
        net.port_names = saved["ports"].tolist()
    net.name = directory.name
    check_network(net)
    expected = (
        np.arange(1, 801) * 0.25e9
        if spec["dense"]
        else np.asarray(spec["anchors_ghz"]) * 1e9
    )
    np.testing.assert_allclose(net.f, expected, rtol=0, atol=1)
    if net.port_names != list(PORTS) or not np.allclose(
        net.z0, Z_SE, rtol=0, atol=1e-12
    ):
        raise ValueError("Archived canonical ports/reference impedances differ")
    raw = aligned_read_network(directory)
    np.testing.assert_allclose(net.f, raw.f, rtol=0, atol=1)
    np.testing.assert_allclose(net.s, raw.s, rtol=0, atol=1e-12)
    return net


def aligned_execute(directory, executable, env, processes, threads, launcher_args=None):
    """Stop this owned job at a failed main GMRES solve; retain its partial logs."""

    def terminate(process):
        # MPI ranks may have their own process groups. Snapshot only descendants
        # of this owned launcher before stopping it; wait for every rank too.
        with suppress(psutil.NoSuchProcess):
            children = psutil.Process(process.pid).children(recursive=True)
            for child in reversed(children):
                with suppress(psutil.NoSuchProcess):
                    child.terminate()
            _, alive = psutil.wait_procs(children, timeout=10)
            for child in alive:
                with suppress(psutil.NoSuchProcess):
                    child.kill()
            psutil.wait_procs(alive, timeout=10)
        if process.poll() is None:
            with suppress(ProcessLookupError):
                os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                with suppress(ProcessLookupError):
                    os.killpg(process.pid, signal.SIGKILL)
                process.wait()

    command = [executable, "-np", str(processes), "-nt", str(threads)]
    if launcher_args:
        command.extend(["--launcher-args", launcher_args])
    command.append("config.json")
    failure = None
    with (directory / "solver.log").open("w") as log:
        process = subprocess.Popen(  # noqa: S603 -- explicit user-selected Palace executable
            command,
            cwd=directory,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
            start_new_session=True,
        )
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                # Estimator PCG messages with MaxIts=0 are deliberately excluded.
                if "GMRES solver did NOT converge" in line:
                    failure = line.strip()
                    terminate(process)
                    break
            code = process.wait()
        except BaseException:
            terminate(process)
            raise
        finally:
            process.stdout.close()
    return {"returncode": code, "linear_failure": failure, "process_group": process.pid}


def aligned_run_case(
    label,
    geometry,
    stack,
    signal_key,
    return_key,
    *,
    gap,
    profile,
    overrides=None,
    section=None,
    slot_centres=None,
    dense=False,
    anchors_ghz=(25.0, 112.5, 200.0),
    provenance=None,
):
    """Run/reload one complete network; no missing column or failed solve is accepted."""
    profile = {
        "axial_end_um": None,
        "axial_end_extent_um": 20.0,
        "positive_preconditioner": True,
        "linear_max_iterations": 1000,
        "linear_restart": 120,
        **profile,
    }
    spec = {
        "schema": 1,
        "geometry_um": asdict(geometry),
        "gap": gap,
        "profile": profile,
        "layers": {
            k: stack.layers[k].model_dump(mode="json") for k in (signal_key, return_key)
        },
        "dielectrics": stack.dielectrics,
        "materials": stack.materials,
        "material_overrides": overrides or {},
        "signal_key": signal_key,
        "return_key": return_key,
        "slot_centres": slot_centres,
        "dense": dense,
        "anchors_ghz": anchors_ghz,
        "palace_version": PALACE_VERSION,
        "versions": VERSIONS,
        "gsim_revision": GSIM_REVISION,
        "provenance": provenance or {},
        "port_names_native": ["p1", "p3", "p2", "p4"],
        "reference_ohm": Z_SE,
        "runtime": {
            "MPI_processes": LOCAL_PROCESSES,
            "OpenMP_threads": ALIGNED_THREADS,
            "OpenBLAS_threads": ALIGNED_BLAS_THREADS,
            "launcher_args": ALIGNED_MPI_ARGS,
        },
        "reference_planes_um": [0.0, geometry.length],
    }
    if section is not None:
        spec["section_sha256"] = hashlib.sha256(
            np.ascontiguousarray(section[0], dtype=np.float64).tobytes()
            + np.ascontiguousarray(section[1], dtype=np.int64).tobytes()
        ).hexdigest()
    signature = hashlib.sha256(json_text(spec).encode()).hexdigest()[:16]
    directory = OUTPUT / f"aligned-{label}-{signature}"
    manifest_path, data_path = directory / "manifest.json", directory / "network.npz"
    if manifest_path.exists() and data_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("status") == "complete":
            assert json_text(manifest["spec"]) == json_text(spec)
            assert (
                hashlib.sha256(data_path.read_bytes()).hexdigest() == manifest["sha256"]
            )
            return aligned_load_archive(directory, manifest)
    if not RUN_ALIGNED:
        return None
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {"status": "incomplete", "spec": spec}
    manifest_path.write_text(json_text(manifest))
    try:
        _mesh, config, summary = aligned_config(
            stack,
            asdict(geometry),
            signal_key,
            return_key,
            directory,
            profile=profile,
            overrides=overrides,
            gap=gap,
            section=section,
            slot_centres=slot_centres,
            frequencies_ghz=anchors_ghz,
            dense=dense,
        )
        validate_generated_config(config)
        manifest["inputs"] = {
            name: hashlib.sha256((directory / name).read_bytes()).hexdigest()
            for name in ("palace.msh", "config.json", "port_information.json")
        }
        manifest["mesh"] = summary
        manifest_path.write_text(json_text(manifest))
        executable = shutil.which(PALACE_EXECUTABLE or "palace")
        if executable is None:
            raise FileNotFoundError("Set PALACE_EXECUTABLE to a local Palace launcher")  # noqa: TRY301 -- record failed experiment
        env = os.environ.copy()
        runtime_lib = Path(executable).resolve().parent.parent / "lib"
        if runtime_lib.is_dir():
            env["LD_LIBRARY_PATH"] = (
                str(runtime_lib) + os.pathsep + env.get("LD_LIBRARY_PATH", "")
            )
        env["OPENBLAS_NUM_THREADS"] = str(ALIGNED_BLAS_THREADS)
        result = aligned_execute(
            directory,
            executable,
            env,
            LOCAL_PROCESSES,
            ALIGNED_THREADS,
            ALIGNED_MPI_ARGS,
        )
        manifest["exit_code"] = result["returncode"]
        if result["linear_failure"] is not None:
            manifest["linear_failure"] = result["linear_failure"]
            raise RuntimeError(  # noqa: TRY301 -- record rejected experiment
                f"Main GMRES failure; partial data rejected in {directory}"
            )
        if result["returncode"]:
            raise RuntimeError(  # noqa: TRY301 -- record rejected experiment
                f"Palace exited {result['returncode']}; inspect {directory}"
            )
        evidence = aligned_check_solver(directory, manifest, dense)
        manifest["solver_evidence"] = asdict(evidence)
        manifest["solver_log_sha256"] = hashlib.sha256(
            (directory / "solver.log").read_bytes()
        ).hexdigest()
        net = aligned_read_network(directory)
        expected_f = (
            np.arange(1, 801) * 0.25e9 if dense else np.asarray(anchors_ghz) * 1e9
        )
        np.testing.assert_allclose(net.f, expected_f, rtol=0, atol=1)
        np.savez_compressed(
            data_path, f=net.f, s=net.s, z0=net.z0, ports=np.asarray(net.port_names)
        )
        net.write_touchstone(filename=str(directory / "network"))
        manifest.update(
            status="complete", sha256=hashlib.sha256(data_path.read_bytes()).hexdigest()
        )
        manifest_path.write_text(json_text(manifest))
    except Exception as exc:
        manifest.update(status="failed", error=f"{type(exc).__name__}: {exc}")
        manifest_path.write_text(json_text(manifest))
        raise
    return net

### 16.5 Run paired geometries

All three geometries share a cross-section and both slot-end landmark sets.
Changing `ALIGNED_PROFILE` selects another discretization and therefore a
different archive. Compare complete networks at matching frequencies; direct
anchors alone cannot establish dense-spectrum or TDR convergence.

In [ ]:
aligned_networks = {}
if RUN_ALIGNED:
    aligned_stack = build_palace_layer_stack(resolved)
    signal, ground = aligned_stack.layers[SIGNAL_KEY], aligned_stack.layers[RETURN_KEY]
    shared_section = aligned_cross_section(
        asdict(GEO),
        (signal.zmin, signal.zmax),
        (ground.zmin, ground.zmax),
        aligned_stack.dielectrics,
        ALIGNED_PROFILE["air_um"],
        ALIGNED_PROFILE["corner_um"],
        ALIGNED_PROFILE["far_um"],
        False,
        False,
        ALIGNED_PROFILE["mirror"],
    )
    for label, geometry, gap in [
        ("intact", GEO, False),
        ("disrupted", GEO, True),
        ("moved", replace(GEO, gap_x=ALIGNED_SLOT_CENTRES[1]), True),
    ]:
        aligned_networks[label] = aligned_run_case(
            label,
            geometry,
            aligned_stack,
            SIGNAL_KEY,
            RETURN_KEY,
            gap=gap,
            profile=ALIGNED_PROFILE,
            overrides=MATERIAL_OVERRIDES,
            section=shared_section,
            slot_centres=ALIGNED_SLOT_CENTRES,
            dense=ALIGNED_DENSE,
            anchors_ghz=ALIGNED_ANCHORS_GHZ,
            provenance={"pdk_source_revision": IHP_REVISION},
        )
else:
    print(
        "Aligned full-network study INCOMPLETE: RUN_ALIGNED=False; pilot columns are not substituted."
    )

### 16.6 Check separation from PDK names

The optional setup fixture changes layer/material names and heights and uses a
single finite dielectric. This is a geometry/configuration check without an EM
field solve.

In [ ]:
# A cheap generic setup check; no electromagnetic field solve is launched.
CHECK_ALIGNED_SETUP = False
if CHECK_ALIGNED_SETUP:
    fixture_stack = build_palace_layer_stack(dummy_stack)
    for layer in fixture_stack.layers.values():
        layer.material = "fixture_metal"
    fixture_stack.materials = {
        "fixture_metal": {"conductivity": 3e7},
        "fixture_glass": {"permittivity": 3.2, "loss_tangent": 0.002},
    }
    fixture_stack.dielectrics = [
        {"name": "bulk", "material": "fixture_glass", "zmin": 0.0, "zmax": 11.0}
    ]
    fixture_geometry = replace(GEO, length=200.0, gap_x=100.0, gap_length=20.0)
    fixture_profile = {
        **ALIGNED_PROFILE,
        "air_um": 30.0,
        "axial_um": 40.0,
        "far_um": 20.0,
    }
    fixture_mesh, fixture_config, fixture_summary = aligned_config(
        fixture_stack,
        asdict(fixture_geometry),
        "trace_sheet",
        "return_sheet",
        OUTPUT / "aligned-synthetic-setup",
        profile=fixture_profile,
        gap=True,
        slot_centres=(100.0, 140.0),
    )
    assert set(fixture_summary["material_ids"]) == {"air", "fixture_glass"}
    assert set(fixture_summary["groups"]["conductor_surfaces"]) == {
        "trace_sheet_xy",
        "trace_sheet_z",
        "return_sheet_xy",
        "return_sheet_z",
    }
    assert all(p["zmin"] == 2.5 and p["zmax"] == 9.0 for p in fixture_mesh.port_info)
    validate_generated_config(fixture_config)
    print(
        "Aligned geometry/configuration accepted different layer/material names and heights."
    )

## 17. Audit complete aligned-network results

A passing source column is not a passing network. Populate the archive table
below only with complete, independently solved four-port experiments. The loader
checks their actual inputs and logs. Each numerical comparison must preserve the
same geometry, material coefficients, port sheets, impedances, reference planes,
and outer boundary. Airbox/substrate model changes belong in a separate study.

For mixed-mode matrices $S_m$ on two discretizations, report

$$e_S=\max_{f,i,j}|S_{m,1}(f)_{ij}-S_{m,2}(f)_{ij}|,$$

and retain the **0.01 absolute complex-S target**. For mode conversion, inspect
only the off-diagonal differential/common blocks $C(S_m)$. Its measured
variation $e_C$ is a separate observable; a large transmission-phase variation
is not automatically a conversion error. Our empirical resolved-effect screen
is

$$\max_f|C(S_{m,\mathrm{gap}}-S_{m,\mathrm{intact}})|>3e_C.$$

This factor of three is a conservative screening choice, not a confidence
interval or an error bound. Also require baseline conversion below the declared
0.01 mesh tolerance. Frequency-approximation agreement is checked against an
independent direct run on the **same mesh**; direct checks on another mesh do
not isolate interpolation error. Interior direct points should be compared with
the actual adaptive training frequencies before describing them as held out.

For TDR, $e_\rho$ is the largest change in the *absolute* reflected-step curves
between dense meshes, across all three geometries. Detect the slot only if its
baseline-subtracted event exceeds $3e_\rho$. Use the same search gate for both
slot positions, then compare their independently extracted arrivals with
$2\tau x/\ell$ and their difference with $2\tau\Delta x/\ell$. No search is
centered on a predicted slot arrival. Direct anchor matrices cannot supply a
TDR convergence check; dense uniformly spaced spectra are required.

The audit also reports DC/window and bandwidth probes. At a smaller bandwidth,
an arrival can become unresolved; that is a limitation to explain, not a pass to
invent. Airbox/substrate sensitivities remain separate from numerical refinement.
The material extrapolation assumption described earlier remains a physical-model
limitation even after numerical checks pass.


An identical archive under two labels is not an independent refinement. The
audit requires an actual dense h-refinement, an order cross-check, and a direct
frequency check on the same mesh. Order checks are explicitly labeled as dense
or sampled anchors. Missing studies and unresolved events prevent an overall
pass. The airbox study checks sensitivity to the artificial outer boundary;
substrate-depth variation is reported as a physical-model sensitivity.


Use the final candidate refinement pairs in this table. Retain rejected coarser
profiles in the historical study tables, with their failed checks visible.


In [ ]:
# Paths are relative to OUTPUT. Keep absent studies as None; do not substitute
# partial source-column pilots or the small pipeline fixture for the 6 mm device.
ALIGNED_ARCHIVES = {
    "reference": {"intact": None, "disrupted": None, "moved": None},
    "mesh_refinement": {"intact": None, "disrupted": None, "moved": None},
    "order_check": {"intact": None, "disrupted": None, "moved": None},
    "direct_check": {"intact": None, "disrupted": None, "moved": None},
}
ALIGNED_MODEL_VARIANTS = {
    "airbox": {"intact": None, "disrupted": None},
    "substrate": {"intact": None, "disrupted": None},
}


def aligned_physical_signature(manifest, directory):
    spec = manifest["spec"]
    config = json.loads((directory / "config.json").read_text())
    groups = manifest["mesh"]["groups"]
    volume_names = {
        value["phys_group"]: name for name, value in groups["volumes"].items()
    }
    conductor_names = {
        value["phys_group"]: name
        for name, value in groups["conductor_surfaces"].items()
    }

    def named(entries, names):
        return sorted(
            (
                names[item["Attributes"][0]],
                {k: v for k, v in item.items() if k != "Attributes"},
            )
            for item in entries
        )

    boundaries = config["Boundaries"]
    return json_text(
        {
            "geometry_um": spec["geometry_um"],
            "gap": spec["gap"],
            "layers": spec["layers"],
            "dielectrics": spec["dielectrics"],
            "air_um": spec["profile"]["air_um"],
            "exterior_only": spec["profile"]["exterior_only"],
            "materials": named(config["Domains"]["Materials"], volume_names),
            "conductors": named(boundaries["Conductivity"], conductor_names),
            "ports": [
                {k: v for k, v in item.items() if k not in {"Attributes", "Excitation"}}
                for item in boundaries["LumpedPort"]
            ],
            "absorbing": {
                k: v for k, v in boundaries["Absorbing"].items() if k != "Attributes"
            },
            "port_geometry": json.loads(
                (directory / "port_information.json").read_text()
            )["ports"],
            "port_order": spec["port_names_native"],
            "reference_planes_um": spec["reference_planes_um"],
            "L0": config["Model"]["L0"],
        }
    )


def aligned_conversion(s):
    return np.concatenate([s[:, :2, 2:].reshape(-1), s[:, 2:, :2].reshape(-1)])


def aligned_common_samples(reference, other):
    # No interpolation can hide an error in a direct-anchor comparison.
    indices = []
    for frequency in other.f:
        found = np.flatnonzero(np.isclose(reference.f, frequency, rtol=0, atol=1))
        if len(found) != 1:
            raise ValueError(
                "Direct checks must coincide with recorded output frequencies"
            )
        indices.append(found[0])
    return np.asarray(indices)


def aligned_measure_refinement(studies, reference="reference", kinds=None):
    """Numerical metrics only; archive identity/physical checks precede this call."""
    mm = {
        study: {case: mixed_mode(net)[1] for case, net in cases.items()}
        for study, cases in studies.items()
    }
    rows = []
    for study, cases in mm.items():
        if study == reference:
            continue
        for case, net in cases.items():
            base = mm[reference][case]
            idx = aligned_common_samples(base, net)
            delta = net.s - base.s[idx]
            dense = len(net.f) == len(base.f) == 800
            rho_change = None
            if dense:
                t0, rho0, _ = tdr(base.f, base.s[:, 0, 0], 2 * Z_SE)
                t1, rho1, _ = tdr(net.f, net.s[:, 0, 0], 2 * Z_SE)
                np.testing.assert_allclose(t0, t1, rtol=0, atol=1e-18)
                rho_change = float(np.max(np.abs(rho1 - rho0)))
            rows.append(
                {
                    "study": study,
                    "kind": (kinds or {}).get(study, "unclassified"),
                    "geometry": case,
                    "scope": "dense" if dense else "sampled anchors",
                    "max complex mixed S change": float(np.max(np.abs(delta))),
                    "max conversion-block change": float(
                        np.max(np.abs(aligned_conversion(delta)))
                    ),
                    "max absolute reflected-step change": rho_change,
                }
            )
    return mm, pd.DataFrame(rows)


def aligned_event_checks(mm, geometry, eps_rho):
    intact, gap, moved = (mm[k] for k in ("intact", "disrupted", "moved"))
    tau = differential_delay(intact)
    resolution_s = 2 / intact.f[-1]
    expected = 2 * tau * geometry.gap_x / geometry.length
    moved_expected = 2 * tau * ALIGNED_SLOT_CENTRES[1] / geometry.length
    arrival = gap_arrival(intact, gap, tau, resolution_s, 3 * eps_rho)
    moved_arrival = gap_arrival(intact, moved, tau, resolution_s, 3 * eps_rho)
    separation = min(expected, 2 * tau - moved_expected) > 3 * resolution_s
    shift = moved_expected - expected
    checks = {
        "gap timing resolved": separation
        and np.isfinite(arrival)
        and abs(arrival - expected) <= resolution_s,
        "moved-slot shift resolved": separation
        and shift > resolution_s
        and np.isfinite(moved_arrival)
        and np.isfinite(arrival)
        and abs((moved_arrival - arrival) - shift) <= resolution_s,
    }

    def finite(x):
        return float(x) if np.isfinite(x) else None

    return checks, {
        "one_way_delay_s": tau,
        "time_resolution_s": resolution_s,
        "gap_expected_s": expected,
        "gap_arrival_s": finite(arrival),
        "moved_expected_s": moved_expected,
        "moved_arrival_s": finite(moved_arrival),
    }


def aligned_tdr_sensitivity(mm, eps_rho):
    intact, gap, moved = (mm[k] for k in ("intact", "disrupted", "moved"))
    tau = differential_delay(intact)
    rows = []
    for window in ("hamming", "hann"):
        for dc in ("linear", "constant"):
            times = []
            for net in (gap, moved):
                t, base, _ = tdr(
                    intact.f, intact.s[:, 0, 0], 2 * Z_SE, window=window, dc=dc
                )
                _, reflection, _ = tdr(
                    net.f, net.s[:, 0, 0], 2 * Z_SE, window=window, dc=dc
                )
                region = (t > 2 * 2 / net.f[-1]) & (t < 2 * tau - 2 * 2 / net.f[-1])
                delta = np.abs(reflection - base)
                peak = delta[region].max() if region.any() else 0.0
                found = np.flatnonzero(region & (delta >= max(3 * eps_rho, 0.5 * peak)))
                times.append(
                    float(t[found[0]]) if peak > 3 * eps_rho and len(found) else None
                )
            rows.append(
                {
                    "window": window,
                    "dc": dc,
                    "gap_arrival_s": times[0],
                    "moved_arrival_s": times[1],
                    "shift_s": times[1] - times[0] if None not in times else None,
                }
            )
    return pd.DataFrame(rows)


def aligned_comparison_kind(manifests, study, reference):
    base = manifests[reference]["intact"]
    other = manifests[study]["intact"]
    p0, p1 = base["spec"]["profile"], other["spec"]["profile"]
    if p0["order"] != p1["order"]:
        return "order crosscheck"
    spatial = ("axial_um", "corner_um", "far_um", "axial_end_um", "axial_end_extent_um")
    if any(p0.get(key) != p1.get(key) for key in spatial):
        if base["inputs"]["palace.msh"] == other["inputs"]["palace.msh"]:
            raise ValueError("Declared spatial refinement did not change the mesh")
        return "h refinement"
    if (
        base["inputs"]["palace.msh"] == other["inputs"]["palace.msh"]
        and not other["spec"]["dense"]
    ):
        anchors = other["spec"]["anchors_ghz"]
        if len(anchors) < 3 or sum(0.25 < f < 200 for f in anchors) < 2:
            raise ValueError(
                "Use at least two interior direct anchors plus a third frequency"
            )
        return "direct frequency check"
    return "other numerical comparison"


def aligned_bandwidth_probe(mm, geometry, eps_rho):
    rows = []
    for cutoff in (100e9, 200e9):
        band = {}
        for case, net in mm.items():
            use = net.f <= cutoff
            band[case] = rf.Network(
                f=net.f[use], s=net.s[use], z0=net.z0[use], f_unit="Hz"
            )
        resolution = 2 / cutoff
        tau = differential_delay(band["intact"])
        expected = 2 * tau * geometry.gap_x / geometry.length
        moved = 2 * tau * ALIGNED_SLOT_CENTRES[1] / geometry.length
        resolvable = (
            min(expected, 2 * tau - moved) > 3 * resolution
            and moved - expected > resolution
        )
        checks, timing = (
            aligned_event_checks(band, geometry, eps_rho) if resolvable else ({}, {})
        )
        rows.append(
            {
                "bandwidth_GHz": cutoff / 1e9,
                "resolution_ps": resolution * 1e12,
                "geometrically_resolvable": bool(resolvable),
                "status": ("resolved" if all(checks.values()) else "inconclusive")
                if resolvable
                else "resolution limited",
                **timing,
            }
        )
    return pd.DataFrame(rows)


def aligned_model_sensitivity(
    variants,
    reference_nets,
    reference_manifests,
    reference_dirs,
    expected_provenance=None,
):
    if set(variants) != {"airbox", "substrate"} or any(
        set(cases) != {"intact", "disrupted"} for cases in variants.values()
    ):
        raise ValueError("Supply both paired airbox and substrate studies")
    rows, missing = [], []
    for study, cases in variants.items():
        for case, path in cases.items():
            if path is None or not (OUTPUT / path / "manifest.json").exists():
                missing.append((study, case))
                continue
            directory = OUTPUT / path
            manifest = json.loads((directory / "manifest.json").read_text())
            if any(
                manifest["spec"].get("provenance", {}).get(key) != value
                for key, value in (expected_provenance or {}).items()
            ):
                raise ValueError("Model-variant source provenance differs")
            numerical_keys = (
                "order",
                "axial_um",
                "corner_um",
                "far_um",
                "mirror",
                "axial_end_um",
                "axial_end_extent_um",
            )
            if any(
                manifest["spec"]["profile"].get(key)
                != reference_manifests[case]["spec"]["profile"].get(key)
                for key in numerical_keys
            ):
                raise ValueError(
                    "Keep spatial controls fixed in model-sensitivity studies"
                )
            net = aligned_load_archive(directory, manifest)
            original = json.loads(
                aligned_physical_signature(
                    reference_manifests[case], reference_dirs[case]
                )
            )
            changed = json.loads(aligned_physical_signature(manifest, directory))
            allowed = "air_um" if study == "airbox" else "dielectrics"
            if original[allowed] == changed[allowed]:
                raise ValueError(f"Model sensitivity did not change {allowed}")
            original.pop(allowed)
            changed.pop(allowed)
            if json_text(original) != json_text(changed):
                raise ValueError("Model sensitivity changed additional physical inputs")
            base_mm, new_mm = mixed_mode(reference_nets[case])[1], mixed_mode(net)[1]
            np.testing.assert_allclose(new_mm.f, base_mm.f, rtol=0, atol=1)
            delta = float(np.max(np.abs(new_mm.s - base_mm.s)))
            rows.append(
                {
                    "model": study,
                    "geometry": case,
                    "max complex mixed S change": delta,
                    "within 0.01": delta < 0.01,
                }
            )
    return pd.DataFrame(rows), missing


def aligned_audit_archives(archives, reference="reference", expected_provenance=None):
    required = {"intact", "disrupted", "moved"}
    if reference not in archives or any(
        set(cases) != required for cases in archives.values()
    ):
        raise ValueError("Every numerical study requires all three stated geometries")
    missing = [
        (study, case)
        for study, cases in archives.items()
        for case, path in cases.items()
        if path is None or not (OUTPUT / path / "manifest.json").exists()
    ]
    if missing:
        print("Aligned acceptance audit INCOMPLETE: missing complete archives", missing)
        return None
    studies, manifests, directories = {}, {}, {}
    for study, cases in archives.items():
        studies[study], manifests[study], directories[study] = {}, {}, {}
        for case, path in cases.items():
            directory = OUTPUT / path
            manifest = json.loads((directory / "manifest.json").read_text())
            spec = manifest["spec"]
            expected_geometry = asdict(GEO)
            if case == "moved":
                expected_geometry["gap_x"] = ALIGNED_SLOT_CENTRES[1]
            if spec["geometry_um"] != expected_geometry:
                raise ValueError(
                    "Archive geometry differs; a small fixture cannot validate the full device"
                )
            if spec["gap"] != (case != "intact"):
                raise ValueError("Archive does not represent its stated geometry")
            if any(
                spec.get("provenance", {}).get(key) != value
                for key, value in (expected_provenance or {}).items()
            ):
                raise ValueError(
                    "Archive source provenance differs from the declared device example"
                )
            studies[study][case] = aligned_load_archive(directory, manifest)
            manifests[study][case], directories[study][case] = manifest, directory
    for study, cases in manifests.items():
        baseline_signature = json.loads(
            aligned_physical_signature(cases["intact"], directories[study]["intact"])
        )
        for key in ("geometry_um", "gap"):
            baseline_signature.pop(key)
        for case, manifest in cases.items():
            if json_text(manifest["spec"]["profile"]) != json_text(
                cases["intact"]["spec"]["profile"]
            ):
                raise ValueError(
                    "Use the same numerical controls within each paired study"
                )
            np.testing.assert_allclose(
                studies[study][case].f, studies[study]["intact"].f, rtol=0, atol=1
            )
            signature = json.loads(
                aligned_physical_signature(manifest, directories[study][case])
            )
            for key in ("geometry_um", "gap"):
                signature.pop(key)
            if json_text(signature) != json_text(baseline_signature):
                raise ValueError(
                    "Paired geometries changed physical inputs besides the intended slot"
                )
    for study, cases in manifests.items():
        for case, manifest in cases.items():
            if aligned_physical_signature(
                manifest, directories[study][case]
            ) != aligned_physical_signature(
                manifests[reference][case], directories[reference][case]
            ):
                raise ValueError(
                    f"Physical model differs in numerical comparison: {study}/{case}"
                )
    kinds = {
        study: aligned_comparison_kind(manifests, study, reference)
        for study in studies
        if study != reference
    }
    mm, table = aligned_measure_refinement(studies, reference, kinds)
    show(table)
    dense = table[table.scope == "dense"]
    if (
        dense.empty
        or not ((table.scope == "dense") & (table.kind == "h refinement")).any()
    ):
        print(
            "Spatial/TDR audit INCOMPLETE: an independent dense h-refinement is required."
        )
        return None
    if len(mm[reference]["intact"].f) != 800:
        raise ValueError("Select a dense reference network")
    eps_s = float(dense["max complex mixed S change"].max())
    eps_cross = float(table["max conversion-block change"].max())
    eps_rho = float(dense["max absolute reflected-step change"].max())
    baseline = float(np.max(np.abs(aligned_conversion(mm[reference]["intact"].s))))
    gap_change = float(
        np.max(
            np.abs(
                aligned_conversion(
                    mm[reference]["disrupted"].s - mm[reference]["intact"].s
                )
            )
        )
    )
    order = table[table.kind == "order crosscheck"]
    direct = table[table.kind == "direct frequency check"]
    event_checks, timing = aligned_event_checks(mm[reference], GEO, eps_rho)
    checks = {
        "dense absolute mixed S change <0.01": eps_s < 0.01,
        "order crosscheck available and within 0.01": not order.empty
        and order["max complex mixed S change"].max() < 0.01,
        "same-mesh direct check available and within 0.01": not direct.empty
        and direct["max complex mixed S change"].max() < 0.01,
        "baseline conversion below mesh tolerance": baseline < 0.01,
        "asymmetric conversion resolved": gap_change > 3 * eps_cross,
        **event_checks,
    }
    sensitivity = aligned_tdr_sensitivity(mm[reference], eps_rho)
    show(sensitivity)
    for column, nominal in [
        ("gap_arrival_s", timing["gap_arrival_s"]),
        ("moved_arrival_s", timing["moved_arrival_s"]),
    ]:
        checks[f"DC/window {column} stable"] = (
            nominal is not None
            and sensitivity[column].notna().all()
            and (sensitivity[column] - nominal).abs().max()
            <= timing["time_resolution_s"]
        )
    bandwidth = aligned_bandwidth_probe(mm[reference], GEO, eps_rho)
    show(bandwidth)
    models, missing_models = aligned_model_sensitivity(
        ALIGNED_MODEL_VARIANTS,
        studies[reference],
        manifests[reference],
        directories[reference],
        expected_provenance,
    )
    show(models)
    air_rows = models[models.model == "airbox"] if not models.empty else pd.DataFrame()
    substrate_rows = (
        models[models.model == "substrate"] if not models.empty else pd.DataFrame()
    )
    checks["airbox sensitivity completed and within 0.01"] = (
        len(air_rows) == 2
        and not any(study == "airbox" for study, _ in missing_models)
        and air_rows["within 0.01"].all()
    )
    checks["substrate sensitivity completed"] = len(substrate_rows) == 2 and not any(
        study == "substrate" for study, _ in missing_models
    )
    # Verify the conversion and inverse on these actual matrices too.
    for cases in studies.values():
        for net in cases.values():
            ordered, converted = mixed_mode(net)
            np.testing.assert_allclose(
                converted.s, M @ ordered.s @ M.T, atol=1e-12, rtol=0
            )
            back = converted.copy()
            back.gmm2se(p=2)
            np.testing.assert_allclose(back.s, ordered.s, atol=1e-12, rtol=0)
    checks["mixed-mode formula and inverse"] = True
    show(pd.Series({k: bool(v) for k, v in checks.items()}, name="Passed").to_frame())
    report = {
        "status": "passed"
        if all(checks.values())
        else "incomplete/inconclusive/failed",
        "scope": "empirical checks on complete recorded networks; anchor agreement is not a uniform order-error bound",
        "checks": {k: bool(v) for k, v in checks.items()},
        "eps_S": eps_s,
        "eps_conversion": eps_cross,
        "eps_reflected_step": eps_rho,
        "baseline_conversion": baseline,
        "gap_conversion_change": gap_change,
        "timing": timing,
        "missing_model_studies": missing_models,
    }
    OUTPUT.mkdir(parents=True, exist_ok=True)
    for name, frame in [
        ("refinement-comparisons", table),
        ("tdr-sensitivity", sensitivity),
        ("bandwidth-probe", bandwidth),
        ("model-sensitivity", models),
    ]:
        frame.to_csv(OUTPUT / f"aligned-{name}.csv", index=False)
    (OUTPUT / "aligned-validation.json").write_text(json_text(report))
    return report


aligned_validation = aligned_audit_archives(
    ALIGNED_ARCHIVES, expected_provenance={"pdk_source_revision": IHP_REVISION}
)